<a href="https://colab.research.google.com/github/speedyhok/-PulseOS-Clinical-Intelligence/blob/main/Causal_Validation_and_Circuit_Tracing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal Validation and Circuit Tracing of an SAE Feature in Llama 3.1 8B

**Author:** Mohibul Hoque  
**Email:** hokworks@gmail.com  

---

## Research Question

> **Can an independently validated semantic interpretation of an SAE feature
> predict its causal effect on language-model behavior, and can that causal
> effect be traced to a local mechanistic circuit?**

## Overview

This notebook investigates Llama 3.1 8B using a pretrained 131k-feature
Sparse Autoencoder (SAE) attached to the Layer-19 residual stream.

The experiment proceeds through behavioral validation, SAE feature discovery,
causal intervention, robustness testing, and local circuit tracing.

In [ ]:
!pip install -q -U sae_lens==6.49.1 transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.1/313.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.7/334.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.7/241.7 kB 9.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

# 1. Experimental Setup

## 1.1 Model and SAE

We use Llama 3.1 8B together with a pretrained 131k-feature sparse
autoencoder (SAE) attached to the Layer-19 residual stream.

No model or SAE parameters are trained in this experiment. All analyses are
performed using inference-time activation measurement and intervention.

## 1.2 Computational Setup

The notebook is designed as an inference-only mechanistic interpretability
experiment and is executed in Google Colab.

## 1.3 Initialization and Sanity Checks

Before conducting the behavioral experiments, we verify model loading,
tokenization, SAE dimensions, and access to the Layer-19 residual stream.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print("Loaded:", MODEL_ID)
print("Hidden size:", model.config.hidden_size)
print("Layers:", model.config.num_hidden_layers)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded: meta-llama/Llama-3.1-8B-Instruct
Hidden size: 4096
Layers: 32


In [ ]:
from sae_lens import SAE

sae = SAE.from_pretrained(
    release="llama-3.1-8b-instruct-andyrdt",
    sae_id="resid_post_layer_19_trainer_1",
    device="cpu",     # keep SAE on CPU; only the activation moves between devices
    dtype="float32",
)
sae.eval()

print("SAE loaded:", sae.cfg.metadata.hook_name if hasattr(sae.cfg, "metadata") else sae.cfg)
print("d_in:", sae.cfg.d_in, " d_sae:", sae.cfg.d_sae)

config.json:   0%|          | 0.00/951 [00:00<?, ?B/s]

resid_post_layer_19/trainer_1/ae.pt: reconstructing file:   0%|          |  0.00B / 4.30GB            

resid_post_layer_19/trainer_1/ae.pt: downloading bytes:           |  0.00B            

SAE loaded: blocks.19.hook_resid_post
d_in: 4096  d_sae: 131072


### Setup Check

The model and SAE are successfully initialized and the Layer-19 residual
stream is accessible for both feature measurement and causal intervention.

Model and SAE Sanity Checks

In [ ]:
import torch.nn.functional as F
import warnings, logging
warnings.filterwarnings("ignore", message=".*will be cast from.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

captured = {}

def capture_layer19(module, inputs, output):
    captured["layer19"] = output[0].detach().float().cpu() if isinstance(output, tuple) else output.detach().float().cpu()

hook = model.model.layers[19].register_forward_hook(capture_layer19)

messages = [{"role": "user", "content": "The dog is"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,        # <-- forces a proper BatchEncoding with .input_ids etc.
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    _ = model(**inputs, use_cache=False)   # <-- unpack with **, not positional

hook.remove()

h = captured["layer19"]
x = h[0, -1, :].unsqueeze(0)

print("Real activation shape:", x.shape)
print("Real residual norm:", x.norm().item())

Real activation shape: torch.Size([1, 4096])
Real residual norm: 12.407608032226562


In [ ]:
with torch.no_grad():
    feature_acts = sae.encode(x)
    reconstruction = sae.decode(feature_acts)

mse = F.mse_loss(reconstruction, x)
relative_error = (reconstruction - x).norm(dim=-1) / x.norm(dim=-1).clamp_min(1e-8)
cosine_similarity = F.cosine_similarity(reconstruction, x, dim=-1)
active_features = (feature_acts > 0).sum(dim=-1)

print("PRETRAINED SAE COMPATIBILITY RESULT")
print("Original norm      :", x.norm().item())
print("Reconstructed norm :", reconstruction.norm().item())
print("MSE                :", mse.item())
print("Relative error     :", relative_error.item())
print("Cosine similarity  :", cosine_similarity.item())
print("Active features    :", active_features.item())

values, indices = torch.topk(feature_acts[0], k=10)
print("\nTop 10 SAE features:")
for rank, (idx, val) in enumerate(zip(indices.tolist(), values.tolist()), 1):
    print(f"{rank:2d}. Feature {idx:6d}  activation = {val:.6f}")

PRETRAINED SAE COMPATIBILITY RESULT
Original norm      : 12.407608032226562
Reconstructed norm : 10.238441467285156
MSE                : 0.009915078990161419
Relative error     : 0.5136178135871887
Cosine similarity  : 0.8586733341217041
Active features    : 35

Top 10 SAE features:
 1. Feature  62198  activation = 3.620737
 2. Feature  91936  activation = 2.626965
 3. Feature   5581  activation = 2.202171
 4. Feature  27967  activation = 1.523650
 5. Feature 131008  activation = 1.473963
 6. Feature  71061  activation = 1.409590
 7. Feature  68935  activation = 1.392628
 8. Feature 109387  activation = 1.105586
 9. Feature  82031  activation = 0.927903
10. Feature  49920  activation = 0.901727


In [ ]:
test_prompts = [
    "The dog is",
    "The cat sat on the",
    "In 1969, the first humans landed on the",
    "She opened the door and saw",
    "The stock market fell sharply after",
]

results = []
for prompt in test_prompts:
    captured = {}
    hook = model.model.layers[19].register_forward_hook(capture_layer19)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        _ = model(**inputs, use_cache=False)
    hook.remove()

    xi = captured["layer19"][0, -1, :].unsqueeze(0)
    with torch.no_grad():
        feats = sae.encode(xi)
        recon = sae.decode(feats)

    cos = F.cosine_similarity(recon, xi, dim=-1).item()
    rel_err = ((recon - xi).norm(dim=-1) / xi.norm(dim=-1)).item()
    n_active = (feats > 0).sum().item()
    results.append((prompt, cos, rel_err, n_active))
    print(f"{prompt!r:45s} cos={cos:.3f}  rel_err={rel_err:.3f}  active={n_active}")

'The dog is'                                  cos=0.859  rel_err=0.514  active=35
'The cat sat on the'                          cos=0.853  rel_err=0.523  active=25
'In 1969, the first humans landed on the'     cos=0.885  rel_err=0.466  active=37
'She opened the door and saw'                 cos=0.866  rel_err=0.501  active=40
'The stock market fell sharply after'         cos=0.875  rel_err=0.484  active=34


In [ ]:
# ============================================================
# IMPORTS — RUN THIS FIRST
# ============================================================

import os
import math
import json
import random
import re
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from collections import Counter, defaultdict

from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Imports loaded successfully.
PyTorch: 2.11.0+cu128
NumPy: 2.0.2
Pandas: 2.2.2


# 2. Behavioral Task: Animal vs Vehicle

We first define the model behavior that will be used as the causal target.

## 2.1 Behavioral Definition

For a word \(w\), we compare the log-probability of two competing
continuations:

$$
M(w) =
\log P(\text{" an animal"} \mid w)
-
\log P(\text{" a vehicle"} \mid w)
$$

Positive values indicate an animal preference, while negative values indicate
a vehicle preference.

## 2.2 Baseline Behavioral Test

Before studying SAE features, we verify that the model reliably distinguishes
the two categories.

## 2.1 Animal-vs-Vehicle Behavioral Variable

In [ ]:
BEHAVIOR = {
    "name": "animal_vehicle",
    "prompt_template": "The {word} is",

    "cat_a": "animal",
    "cont_a": " an animal",

    "cat_b": "vehicle",
    "cont_b": " a vehicle",

    "discovery_a": [
        "dog", "cat", "horse", "eagle", "dolphin",
        "tiger", "rabbit", "cow", "lion", "whale"
    ],

    "discovery_b": [
        "car", "truck", "bus", "airplane", "train",
        "bicycle", "motorcycle", "boat", "van", "helicopter"
    ],

    "heldout_a": [
        "elephant", "zebra", "monkey", "bear", "giraffe",
        "penguin", "shark", "snake", "sheep", "goat"
    ],

    "heldout_b": [
        "taxi", "scooter", "tractor", "subway", "ship",
        "jeep", "rocket", "automobile", "canoe", "yacht"
    ],
}

overlap = (
    set(BEHAVIOR["discovery_a"]) |
    set(BEHAVIOR["discovery_b"])
) & (
    set(BEHAVIOR["heldout_a"]) |
    set(BEHAVIOR["heldout_b"])
)

assert not overlap, f"Leakage between discovery and held-out: {overlap}"

print(
    "Behavior:",
    BEHAVIOR["name"],
    "— no discovery/held-out overlap. OK."
)

Behavior: animal_vehicle — no discovery/held-out overlap. OK.


In [ ]:
import torch
import torch.nn.functional as F


def make_prompt(word):
    return BEHAVIOR["prompt_template"].format(word=word)


def get_layer19_activation(word):
    """Real Layer-19 residual-post activation, last token, chat-templated."""

    captured = {}

    def hook_fn(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["h"] = hidden.detach().float().cpu()

    handle = model.model.layers[19].register_forward_hook(hook_fn)

    try:
        messages = [
            {"role": "user", "content": make_prompt(word)}
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            _ = model(**inputs, use_cache=False)

    finally:
        handle.remove()

    return captured["h"][0, -1, :].unsqueeze(0)


def seq_logprob(prompt, continuation):
    """log P(continuation | prompt), chat-templated prefix."""

    messages = [
        {"role": "user", "content": prompt}
    ]

    prefix_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )

    if not torch.is_tensor(prefix_ids):
        prefix_ids = prefix_ids.input_ids

    prefix_ids = prefix_ids.to(model.device)

    cont_ids = tokenizer(
        continuation,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    full_ids = torch.cat(
        [prefix_ids, cont_ids],
        dim=1
    )

    with torch.no_grad():
        logits = model(full_ids).logits

    start = prefix_ids.shape[1] - 1
    end = full_ids.shape[1] - 1

    log_probs = F.log_softmax(
        logits[0, start:end],
        dim=-1
    )

    token_logps = log_probs.gather(
        1,
        cont_ids[0].unsqueeze(-1)
    ).squeeze(-1)

    return token_logps.sum().item()


def behavioral_margin(word):
    """
    M = logP(cont_a) - logP(cont_b)

    Positive -> model favors category A
    Negative -> model favors category B
    """

    prompt = make_prompt(word)

    lp_a = seq_logprob(
        prompt,
        BEHAVIOR["cont_a"]
    )

    lp_b = seq_logprob(
        prompt,
        BEHAVIOR["cont_b"]
    )

    return lp_a - lp_b

## 2.2 Baseline Behavioral Validation

In [ ]:
results = []

for word in BEHAVIOR["discovery_a"] + BEHAVIOR["discovery_b"]:
    m = behavioral_margin(word)

    label = (
        BEHAVIOR["cat_a"]
        if word in BEHAVIOR["discovery_a"]
        else BEHAVIOR["cat_b"]
    )

    correct = (
        (m > 0)
        if label == BEHAVIOR["cat_a"]
        else (m < 0)
    )

    results.append(
        (word, label, m, correct)
    )

    print(
        f"{word:12s} "
        f"[{label:7s}] "
        f"M = {m:+.3f} "
        f"{'OK' if correct else 'WRONG'}"
    )

acc = sum(r[3] for r in results) / len(results)

print(f"\nAccuracy: {acc:.0%}")

if acc < 0.9:
    print(
        "WARNING: accuracy is low. "
        "This behavior may not be clean enough to use."
    )

dog          [animal ] M = +13.750 OK
cat          [animal ] M = +13.375 OK
horse        [animal ] M = +12.500 OK
eagle        [animal ] M = +12.250 OK
dolphin      [animal ] M = +14.750 OK
tiger        [animal ] M = +14.750 OK
rabbit       [animal ] M = +7.625 OK
cow          [animal ] M = +14.625 OK
lion         [animal ] M = +16.625 OK
whale        [animal ] M = +10.750 OK
car          [vehicle] M = -8.625 OK
truck        [vehicle] M = -10.000 OK
bus          [vehicle] M = -4.500 OK
airplane     [vehicle] M = -5.625 OK
train        [vehicle] M = -2.375 OK
bicycle      [vehicle] M = -9.125 OK
motorcycle   [vehicle] M = -8.000 OK
boat         [vehicle] M = -2.875 OK
van          [vehicle] M = -11.375 OK
helicopter   [vehicle] M = -5.500 OK

Accuracy: 100%


### Result

The model exhibits a strong animal-vs-vehicle behavioral distinction, making
this a suitable target for causal intervention.

# 3. Feature Discovery

We search the SAE for features whose activation differs between animal and
vehicle examples.

Feature discovery is treated as hypothesis generation rather than causal
evidence.

In [ ]:
# ============================================================
# SAE FEATURE EXTRACTION + CANDIDATE DISCOVERY
# ============================================================

discovery_words = (
    [(w, BEHAVIOR["cat_a"]) for w in BEHAVIOR["discovery_a"]]
    + [(w, BEHAVIOR["cat_b"]) for w in BEHAVIOR["discovery_b"]]
)

feature_matrix = []
labels = []

for word, label in discovery_words:

    # Real Layer-19 residual-post activation
    x = get_layer19_activation(word)

    # Encode through the pretrained 131k SAE
    with torch.no_grad():
        feats = sae.encode(x)

    feature_matrix.append(feats.squeeze(0))
    labels.append(label)

    print(
        f"{word:12s} "
        f"[{label:7s}] "
        f"active features: {(feats > 0).sum().item()}"
    )

feature_matrix = torch.stack(feature_matrix)

print("\nFeature matrix:", tuple(feature_matrix.shape))
print("Expected:", (20, 131072))


# ============================================================
# Split by behavior
# ============================================================

a_mask = torch.tensor(
    [label == BEHAVIOR["cat_a"] for label in labels],
    dtype=torch.bool
)

b_mask = torch.tensor(
    [label == BEHAVIOR["cat_b"] for label in labels],
    dtype=torch.bool
)

a_acts = feature_matrix[a_mask]
b_acts = feature_matrix[b_mask]

print("Category A matrix:", tuple(a_acts.shape))
print("Category B matrix:", tuple(b_acts.shape))


# ============================================================
# Feature statistics
# ============================================================

a_mean = a_acts.mean(dim=0)
b_mean = b_acts.mean(dim=0)

# A positive value means stronger activation for category A
diff = a_mean - b_mean

# Activation frequency
a_freq = (a_acts > 0).float().mean(dim=0)
b_freq = (b_acts > 0).float().mean(dim=0)


# ============================================================
# Candidate ranking
# ============================================================

# Strong A candidates:
#   large positive difference
#   frequent in A
#   rare in B
a_score = diff * a_freq * (1 - b_freq)

# Strong B candidates:
#   large negative difference
#   frequent in B
#   rare in A
b_score = (-diff) * b_freq * (1 - a_freq)


top_a = torch.topk(a_score, k=10).indices
top_b = torch.topk(b_score, k=10).indices


# ============================================================
# Display top A candidates
# ============================================================

print(
    f"\nTOP {BEHAVIOR['cat_a'].upper()} CANDIDATES"
)

for idx in top_a.tolist():

    print(
        f"F{idx:6d}  "
        f"A_mean={a_mean[idx]:7.3f}  "
        f"B_mean={b_mean[idx]:7.3f}  "
        f"diff={diff[idx]:7.3f}  "
        f"A_freq={a_freq[idx]:.1%}  "
        f"B_freq={b_freq[idx]:.1%}  "
        f"score={a_score[idx]:7.3f}"
    )


# ============================================================
# Display top B candidates
# ============================================================

print(
    f"\nTOP {BEHAVIOR['cat_b'].upper()} CANDIDATES"
)

for idx in top_b.tolist():

    print(
        f"F{idx:6d}  "
        f"A_mean={a_mean[idx]:7.3f}  "
        f"B_mean={b_mean[idx]:7.3f}  "
        f"diff={diff[idx]:7.3f}  "
        f"A_freq={a_freq[idx]:.1%}  "
        f"B_freq={b_freq[idx]:.1%}  "
        f"score={b_score[idx]:7.3f}"
    )

dog          [animal ] active features: 35
cat          [animal ] active features: 36
horse        [animal ] active features: 38
eagle        [animal ] active features: 36
dolphin      [animal ] active features: 36
tiger        [animal ] active features: 37
rabbit       [animal ] active features: 42
cow          [animal ] active features: 32
lion         [animal ] active features: 37
whale        [animal ] active features: 38
car          [vehicle] active features: 42
truck        [vehicle] active features: 30
bus          [vehicle] active features: 39
airplane     [vehicle] active features: 38
train        [vehicle] active features: 37
bicycle      [vehicle] active features: 44
motorcycle   [vehicle] active features: 35
boat         [vehicle] active features: 32
van          [vehicle] active features: 34
helicopter   [vehicle] active features: 38

Feature matrix: (20, 131072)
Expected: (20, 131072)
Category A matrix: (10, 131072)
Category B matrix: (10, 131072)

TOP ANIMAL CANDIDATES


# 4. Held-Out Feature Validation

The candidate features identified during discovery are evaluated on a separate
held-out dataset.

This prevents us from selecting a feature solely because it fits the examples
used during discovery.

A feature is carried forward only when its category selectivity generalizes
to unseen examples.

In [ ]:
# ============================================================
# HELD-OUT FEATURE EXTRACTION
# ============================================================

heldout_words = (
    [(w, BEHAVIOR["cat_a"]) for w in BEHAVIOR["heldout_a"]]
    + [(w, BEHAVIOR["cat_b"]) for w in BEHAVIOR["heldout_b"]]
)

heldout_feature_matrix = []

for word, label in heldout_words:

    x = get_layer19_activation(word)

    with torch.no_grad():
        feats = sae.encode(x)

    heldout_feature_matrix.append(feats.squeeze(0))

heldout_feature_matrix = torch.stack(heldout_feature_matrix)

print(
    "Held-out feature matrix:",
    tuple(heldout_feature_matrix.shape)
)

# ============================================================
# Masks
# ============================================================

ho_a_mask = torch.tensor(
    [label == BEHAVIOR["cat_a"] for _, label in heldout_words],
    dtype=torch.bool
)

ho_b_mask = torch.tensor(
    [label == BEHAVIOR["cat_b"] for _, label in heldout_words],
    dtype=torch.bool
)


# ============================================================
# Generalization report
# ============================================================

def report_generalization(feature_id, name):
    feature_id = int(feature_id)

    on_a = heldout_feature_matrix[ho_a_mask, feature_id]
    on_b = heldout_feature_matrix[ho_b_mask, feature_id]

    a_mean = on_a.mean().item()
    b_mean = on_b.mean().item()

    a_active = (on_a > 0).float().mean().item()
    b_active = (on_b > 0).float().mean().item()

    diff = a_mean - b_mean

    print(f"\nFeature {feature_id} ({name})")
    print(
        f"  held-out {BEHAVIOR['cat_a']}: "
        f"mean={a_mean:.3f}, "
        f"active={a_active:.0%}"
    )
    print(
        f"  held-out {BEHAVIOR['cat_b']}: "
        f"mean={b_mean:.3f}, "
        f"active={b_active:.0%}"
    )
    print(f"  held-out difference: {diff:+.3f}")


# ============================================================
# Inspect top candidates
# ============================================================

print("\n" + "=" * 60)
print("TOP ANIMAL CANDIDATES — HELD-OUT")
print("=" * 60)

for idx in top_a[:5]:
    report_generalization(
        idx,
        BEHAVIOR["cat_a"]
    )


print("\n" + "=" * 60)
print("TOP VEHICLE CANDIDATES — HELD-OUT")
print("=" * 60)

for idx in top_b[:5]:
    report_generalization(
        idx,
        BEHAVIOR["cat_b"]
    )

Held-out feature matrix: (20, 131072)

TOP ANIMAL CANDIDATES — HELD-OUT

Feature 37378 (animal)
  held-out animal: mean=0.399, active=80%
  held-out vehicle: mean=0.046, active=10%
  held-out difference: +0.353

Feature 54316 (animal)
  held-out animal: mean=0.491, active=50%
  held-out vehicle: mean=0.000, active=0%
  held-out difference: +0.491

Feature 35049 (animal)
  held-out animal: mean=0.411, active=80%
  held-out vehicle: mean=0.443, active=90%
  held-out difference: -0.032

Feature 123229 (animal)
  held-out animal: mean=0.057, active=10%
  held-out vehicle: mean=0.000, active=0%
  held-out difference: +0.057

Feature 112041 (animal)
  held-out animal: mean=0.313, active=70%
  held-out vehicle: mean=0.405, active=90%
  held-out difference: -0.091

TOP VEHICLE CANDIDATES — HELD-OUT

Feature 57774 (vehicle)
  held-out animal: mean=0.247, active=50%
  held-out vehicle: mean=0.549, active=80%
  held-out difference: -0.302

Feature 110833 (vehicle)
  held-out animal: mean=0.000, a

### Result

F54316 remains strongly animal-associated on held-out examples, while several
other high-ranking discovery candidates fail to generalize.

# 5. Causal Intervention on F54316

We now test whether the selected SAE feature has a causal effect on the
animal-vs-vehicle behavior.

The causal evaluation uses an independent vocabulary that was not used for
feature discovery or held-out validation.

## 5.1 Independent Test Set

A separate set of animal and vehicle words is reserved for causal testing.

## 5.2 Leakage Check

We explicitly verify that the causal-test vocabulary does not overlap with
the discovery or held-out datasets.

## 5.3 Baseline on the Causal-Test Set

We first measure the behavioral margin without intervention.

## 5.4 Feature Intervention

We intervene directly on the Layer-19 residual stream along the SAE decoder
direction for F54316.

The intervention magnitude is calibrated relative to the residual-stream norm
so that different examples are tested on a common scale.

In [ ]:
CAUSAL_TEST = {
    "animal": [
        "camel",
        "kangaroo",
        "gorilla",
        "panda",
        "wolf",
        "fox",
        "deer",
        "otter",
        "koala",
        "badger",
    ],

    "vehicle": [
        "limousine",
        "ferry",
        "tram",
        "pickup",
        "minivan",
        "motorbike",
        "sailboat",
        "glider",
        "convertible",
        "sedan",
    ],
}

print(
    "Causal animals:",
    len(CAUSAL_TEST["animal"])
)

print(
    "Causal vehicles:",
    len(CAUSAL_TEST["vehicle"])
)

Causal animals: 10
Causal vehicles: 10


In [ ]:
# ============================================================
# CAUSAL TEST LEAKAGE CHECK
# ============================================================

used_words = (
    set(BEHAVIOR["discovery_a"])
    | set(BEHAVIOR["discovery_b"])
    | set(BEHAVIOR["heldout_a"])
    | set(BEHAVIOR["heldout_b"])
)

causal_words = (
    set(CAUSAL_TEST["animal"])
    | set(CAUSAL_TEST["vehicle"])
)

overlap = used_words & causal_words

print("Overlap:", overlap)

assert not overlap, (
    f"Causal-test leakage detected: {overlap}"
)

print("Causal test set is clean.")

Overlap: set()
Causal test set is clean.


In [ ]:
# ============================================================
# CAUSAL-TEST BASELINE MARGINS
# ============================================================

causal_words = (
    [(w, BEHAVIOR["cat_a"]) for w in CAUSAL_TEST["animal"]]
    + [(w, BEHAVIOR["cat_b"]) for w in CAUSAL_TEST["vehicle"]]
)

causal_baselines = {}

for word, label in causal_words:

    m = behavioral_margin(word)

    causal_baselines[word] = m

    correct = (
        (m > 0)
        if label == BEHAVIOR["cat_a"]
        else (m < 0)
    )

    print(
        f"{word:12s} "
        f"[{label:7s}] "
        f"M={m:+.3f} "
        f"{'OK' if correct else 'WRONG'}"
    )

causal_accuracy = sum(
    (
        (m > 0)
        if label == BEHAVIOR["cat_a"]
        else (m < 0)
    )
    for word, label in causal_words
    for m in [causal_baselines[word]]
) / len(causal_words)

print(
    f"\nCausal-test baseline accuracy: "
    f"{causal_accuracy:.0%}"
)

camel        [animal ] M=+9.500 OK
kangaroo     [animal ] M=+10.250 OK
gorilla      [animal ] M=+14.750 OK
panda        [animal ] M=+15.375 OK
wolf         [animal ] M=+15.500 OK
fox          [animal ] M=+9.625 OK
deer         [animal ] M=+11.375 OK
otter        [animal ] M=+15.250 OK
koala        [animal ] M=+14.375 OK
badger       [animal ] M=+16.125 OK
limousine    [vehicle] M=-7.375 OK
ferry        [vehicle] M=-3.875 OK
tram         [vehicle] M=-2.750 OK
pickup       [vehicle] M=-11.250 OK
minivan      [vehicle] M=-9.875 OK
motorbike    [vehicle] M=-8.000 OK
sailboat     [vehicle] M=-3.750 OK
glider       [vehicle] M=-2.250 OK
convertible  [vehicle] M=-10.000 OK
sedan        [vehicle] M=-6.000 OK

Causal-test baseline accuracy: 100%


In [ ]:
# ============================================================
# CAUSAL-TEST SAE ACTIVATION CHECK
# ============================================================

CAUSAL_FEATURES = {
    37378: BEHAVIOR["cat_a"],
    54316: BEHAVIOR["cat_a"],
    80609: BEHAVIOR["cat_a"],

    75101: BEHAVIOR["cat_b"],
    110833: BEHAVIOR["cat_b"],
    57174: BEHAVIOR["cat_b"],
}

causal_feature_rows = []

for word, label in causal_words:

    x = get_layer19_activation(word)

    with torch.no_grad():
        feats = sae.encode(x).squeeze(0)

    row = {
        "word": word,
        "label": label,
    }

    for fid in CAUSAL_FEATURES:
        value = feats[fid].item()

        row[f"F{fid}"] = value
        row[f"F{fid}_active"] = value > 0

    causal_feature_rows.append(row)


causal_feature_df = pd.DataFrame(
    causal_feature_rows
)

display(causal_feature_df)

,word,label,F37378,F37378_active,F54316,F54316_active,F80609,F80609_active,F75101,F75101_active,F110833,F110833_active,F57174,F57174_active
0,camel,animal,0.000000,False,1.143841,True,0.418288,True,0.000000,False,0.438451,True,0.0,False
1,kangaroo,animal,0.000000,False,1.358682,True,0.000000,False,0.000000,False,0.000000,False,0.0,False
2,gorilla,animal,0.422181,True,0.634425,True,0.493636,True,0.000000,False,0.000000,False,0.0,False
3,panda,animal,0.000000,False,1.312019,True,0.000000,False,0.000000,False,0.000000,False,0.0,False
4,wolf,animal,0.483663,True,0.000000,False,0.456198,True,0.000000,False,0.000000,False,0.0,False
5,fox,animal,0.490814,True,0.498656,True,0.457784,True,0.000000,False,0.000000,False,0.0,False
6,deer,animal,0.525949,True,0.000000,False,0.000000,False,0.546808,True,0.458607,True,0.0,False
7,otter,animal,0.000000,False,1.292308,True,0.000000,False,0.000000,False,0.000000,False,0.0,False
8,koala,animal,0.000000,False,1.591206,True,0.000000,False,0.000000,False,0.000000,False,0.0,False
9,badger,animal,0.000000,False,1.226940,True,0.000000,False,0.000000,False,0.000000,False,0.0,False


In [ ]:
# ============================================================
# CAUSAL-TEST FEATURE GENERALIZATION
# ============================================================

for fid, feature_label in CAUSAL_FEATURES.items():

    a = causal_feature_df[
        causal_feature_df["label"] == BEHAVIOR["cat_a"]
    ][f"F{fid}"]

    b = causal_feature_df[
        causal_feature_df["label"] == BEHAVIOR["cat_b"]
    ][f"F{fid}"]

    print(
        f"\nF{fid} ({feature_label})"
    )

    print(
        f"  animals : "
        f"mean={a.mean():.3f}, "
        f"active={(a > 0).mean():.0%}"
    )

    print(
        f"  vehicles: "
        f"mean={b.mean():.3f}, "
        f"active={(b > 0).mean():.0%}"
    )

    print(
        f"  difference = "
        f"{a.mean() - b.mean():+.3f}"
    )


F37378 (animal)
  animals : mean=0.192, active=40%
  vehicles: mean=0.123, active=30%
  difference = +0.069

F54316 (animal)
  animals : mean=0.906, active=80%
  vehicles: mean=0.048, active=10%
  difference = +0.857

F80609 (animal)
  animals : mean=0.183, active=40%
  vehicles: mean=0.043, active=10%
  difference = +0.139

F75101 (vehicle)
  animals : mean=0.055, active=10%
  vehicles: mean=0.196, active=40%
  difference = -0.141

F110833 (vehicle)
  animals : mean=0.090, active=20%
  vehicles: mean=0.325, active=60%
  difference = -0.235

F57174 (vehicle)
  animals : mean=0.000, active=0%
  vehicles: mean=0.000, active=0%
  difference = +0.000


In [ ]:
# ============================================================
# FINAL UNSEEN CAUSAL-INTERVENTION SET
# ============================================================

FINAL_CAUSAL_TEST = {
    "animal": [
        "leopard",
        "cheetah",
        "hyena",
        "rhinoceros",
        "hippopotamus",
        "buffalo",
        "antelope",
        "squirrel",
        "flamingo",
        "meerkat",
    ],

    "vehicle": [
        "wagon",
        "carriage",
        "trolley",
        "motorcoach",
        "tanker",
        "ambulance",
        "firetruck",
        "bulldozer",
        "airship",
        "roadster",
    ],
}

print(
    "Final animals:",
    len(FINAL_CAUSAL_TEST["animal"])
)

print(
    "Final vehicles:",
    len(FINAL_CAUSAL_TEST["vehicle"])
)

Final animals: 10
Final vehicles: 10


In [ ]:
# ============================================================
# LEAKAGE CHECK
# ============================================================

used_words = (
    set(BEHAVIOR["discovery_a"])
    | set(BEHAVIOR["discovery_b"])
    | set(BEHAVIOR["heldout_a"])
    | set(BEHAVIOR["heldout_b"])
    | set(CAUSAL_TEST["animal"])
    | set(CAUSAL_TEST["vehicle"])
)

final_words = (
    set(FINAL_CAUSAL_TEST["animal"])
    | set(FINAL_CAUSAL_TEST["vehicle"])
)

overlap = used_words & final_words

print("Overlap:", overlap)

assert not overlap, (
    f"Leakage detected: {overlap}"
)

print("Final causal intervention set is clean.")

Overlap: set()
Final causal intervention set is clean.


In [ ]:
# ============================================================
# FINAL CAUSAL TEST — BASELINE ONLY
# ============================================================

final_causal_words = (
    [
        (w, BEHAVIOR["cat_a"])
        for w in FINAL_CAUSAL_TEST["animal"]
    ]
    +
    [
        (w, BEHAVIOR["cat_b"])
        for w in FINAL_CAUSAL_TEST["vehicle"]
    ]
)

final_baselines = {}

for word, label in final_causal_words:

    m = behavioral_margin(word)

    final_baselines[word] = m

    correct = (
        m > 0
        if label == BEHAVIOR["cat_a"]
        else m < 0
    )

    print(
        f"{word:14s} "
        f"[{label:7s}] "
        f"M={m:+.3f} "
        f"{'OK' if correct else 'WRONG'}"
    )

final_accuracy = sum(
    (
        m > 0
        if label == BEHAVIOR["cat_a"]
        else m < 0
    )
    for word, label in final_causal_words
    for m in [final_baselines[word]]
) / len(final_causal_words)

print(
    f"\nFinal causal-test baseline accuracy: "
    f"{final_accuracy:.0%}"
)

leopard        [animal ] M=+12.375 OK
cheetah        [animal ] M=+11.250 OK
hyena          [animal ] M=+12.250 OK
rhinoceros     [animal ] M=+15.750 OK
hippopotamus   [animal ] M=+17.875 OK
buffalo        [animal ] M=+13.750 OK
antelope       [animal ] M=+12.625 OK
squirrel       [animal ] M=+10.750 OK
flamingo       [animal ] M=+6.375 OK
meerkat        [animal ] M=+15.500 OK
wagon          [vehicle] M=-5.000 OK
carriage       [vehicle] M=-5.250 OK
trolley        [vehicle] M=+0.250 WRONG
motorcoach     [vehicle] M=-7.375 OK
tanker         [vehicle] M=-4.000 OK
ambulance      [vehicle] M=-11.375 OK
firetruck      [vehicle] M=-9.125 OK
bulldozer      [vehicle] M=-1.375 OK
airship        [vehicle] M=-1.625 OK
roadster       [vehicle] M=-10.000 OK

Final causal-test baseline accuracy: 95%


In [ ]:
# ============================================================
# FINAL CAUSAL TEST SETUP
# ============================================================

FEATURE_A = 54316    # animal
FEATURE_B = 110833   # vehicle

FEATURES = {
    FEATURE_A: BEHAVIOR["cat_a"],
    FEATURE_B: BEHAVIOR["cat_b"],
}


# ------------------------------------------------------------
# Compute intervention scale from discovery activations
# ------------------------------------------------------------

feature_alpha = {}

for fid, label in FEATURES.items():

    if label == BEHAVIOR["cat_a"]:
        values = a_acts[:, fid]
    else:
        values = b_acts[:, fid]

    active = values[values > 0]

    if len(active) == 0:
        raise ValueError(
            f"Feature {fid} has no active discovery examples."
        )

    feature_alpha[fid] = float(
        active.mean()
    )

    print(
        f"Feature {fid} ({label}) "
        f"mean active alpha = "
        f"{feature_alpha[fid]:.4f}"
    )


# ------------------------------------------------------------
# Original SAE decoder directions
# ------------------------------------------------------------

decoder_directions = {
    fid: sae.W_dec[fid]
        .detach()
        .clone()
        .float()
    for fid in FEATURES
}


# ------------------------------------------------------------
# Reference Layer-19 residual norm
# ------------------------------------------------------------

reference_word = "leopard"

reference_x = get_layer19_activation(
    reference_word
)

ref_norm = reference_x.norm().item()

print(
    "\nReference word:",
    reference_word
)

print(
    "Reference residual norm:",
    ref_norm
)

for fid in FEATURES:

    alpha = feature_alpha[fid]

    print(
        f"F{fid}: "
        f"alpha={alpha:.4f}, "
        f"2x alpha / residual norm="
        f"{(2 * alpha / ref_norm):.2%}"
    )

Feature 54316 (animal) mean active alpha = 0.7134
Feature 110833 (vehicle) mean active alpha = 0.6097

Reference word: leopard
Reference residual norm: 12.733534812927246
F54316: alpha=0.7134, 2x alpha / residual norm=11.20%
F110833: alpha=0.6097, 2x alpha / residual norm=9.58%


In [ ]:
# ============================================================
# CELL 36 — FINAL CAUSAL HELPERS + INITIAL INTERVENTION TEST
# ============================================================

import numpy as np
import pandas as pd
import torch


# ============================================================
# IMPORTANT
# ============================================================
# DO NOT redefine FEATURE_A / FEATURE_B here.
#
# Cell 35 already defines:
#
#   FEATURE_A = 54316   # animal
#   FEATURE_B = 110833  # vehicle
#
# Cell 36 must use those final features.
# ============================================================

assert "FEATURES" in globals(), (
    "FEATURES is missing. Run Cell 35 first."
)

assert 54316 in FEATURES, (
    "F54316 is missing. Run Cell 35 first."
)

assert 110833 in FEATURES, (
    "F110833 is missing. Run Cell 35 first."
)


# ============================================================
# Estimate typical active SAE activation
# ============================================================

feature_alpha = {}

for fid, label in FEATURES.items():

    if label == BEHAVIOR["cat_a"]:
        values = a_acts[:, fid]
    else:
        values = b_acts[:, fid]

    active = values[values > 0]

    if len(active) == 0:
        raise ValueError(
            f"Feature {fid} has no active discovery examples."
        )

    feature_alpha[fid] = float(
        active.mean()
    )

    print(
        f"Feature {fid} ({label}): "
        f"mean active SAE activation = "
        f"{feature_alpha[fid]:.4f}"
    )


# ============================================================
# ORIGINAL SAE DECODER DIRECTIONS
# ============================================================

decoder_directions = {
    fid: sae.W_dec[fid]
        .detach()
        .clone()
        .float()
    for fid in FEATURES
}


# ============================================================
# HOOKED SEQUENCE LOG-PROBABILITY
# ============================================================

def hooked_seq_logprob(
    prompt,
    continuation,
    feature_id=None,
    strength=0.0,
    sign=1.0,
):
    """
    Compute:

        log P(continuation | prompt)

    Optionally intervene on a Layer-19 SAE feature:

        x' = x + sign * strength * W_dec[feature_id]

    at the final prompt token.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    prefix_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )

    if not torch.is_tensor(prefix_ids):
        prefix_ids = prefix_ids.input_ids

    prefix_ids = prefix_ids.to(model.device)

    cont_ids = tokenizer(
        continuation,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    full_ids = torch.cat(
        [
            prefix_ids,
            cont_ids
        ],
        dim=1
    )

    prefix_len = prefix_ids.shape[1]

    # Last token belonging to the prompt
    target_pos = prefix_len - 1

    handle = None

    # --------------------------------------------------------
    # Feature intervention
    # --------------------------------------------------------

    if feature_id is not None:

        feature_id = int(feature_id)

        if feature_id not in decoder_directions:
            raise KeyError(
                f"Feature {feature_id} not available. "
                f"Available features: "
                f"{list(decoder_directions.keys())}"
            )

        decoder = decoder_directions[
            feature_id
        ].to(
            device=model.device,
            dtype=next(model.parameters()).dtype,
        )

        def post_hook(
            module,
            inputs,
            output,
        ):

            hidden = (
                output[0]
                if isinstance(output, tuple)
                else output
            )

            modified = hidden.clone()

            modified[:, target_pos, :] = (
                modified[:, target_pos, :]
                + sign * strength * decoder
            )

            if isinstance(output, tuple):
                return (
                    (modified,)
                    + output[1:]
                )

            return modified

        handle = (
            model.model.layers[19]
            .register_forward_hook(
                post_hook
            )
        )

    try:

        with torch.no_grad():

            logits = model(
                input_ids=full_ids,
                use_cache=False
            ).logits

    finally:

        if handle is not None:
            handle.remove()

    # --------------------------------------------------------
    # Continuation log-probability
    # --------------------------------------------------------

    start = prefix_len - 1
    end = full_ids.shape[1] - 1

    log_probs = torch.log_softmax(
        logits[0, start:end],
        dim=-1
    )

    token_logps = log_probs.gather(
        1,
        cont_ids[0].unsqueeze(-1)
    ).squeeze(-1)

    return token_logps.sum().item()


# ============================================================
# BEHAVIORAL MARGIN
# ============================================================

def intervention_margin(
    word,
    feature_id=None,
    multiplier=0.0,
    sign=1.0,
):
    """
    M = logP(category A) - logP(category B)

    multiplier is expressed in units of feature_alpha.
    """

    prompt = make_prompt(word)

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    if feature_id is None:

        lp_a = hooked_seq_logprob(
            prompt,
            BEHAVIOR["cont_a"]
        )

        lp_b = hooked_seq_logprob(
            prompt,
            BEHAVIOR["cont_b"]
        )

        return lp_a - lp_b

    # --------------------------------------------------------
    # Intervention
    # --------------------------------------------------------

    feature_id = int(feature_id)

    if feature_id not in feature_alpha:
        raise KeyError(
            f"Feature {feature_id} missing from feature_alpha."
        )

    strength = (
        feature_alpha[feature_id]
        * multiplier
    )

    lp_a = hooked_seq_logprob(
        prompt,
        BEHAVIOR["cont_a"],
        feature_id=feature_id,
        strength=strength,
        sign=sign,
    )

    lp_b = hooked_seq_logprob(
        prompt,
        BEHAVIOR["cont_b"],
        feature_id=feature_id,
        strength=strength,
        sign=sign,
    )

    return lp_a - lp_b


# ============================================================
# SANITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL CAUSAL HELPERS READY")
print("=" * 70)

print(
    "FEATURES:",
    FEATURES
)

print(
    "decoder_directions:",
    list(decoder_directions.keys())
)

print(
    "feature_alpha:",
    {
        fid: round(alpha, 4)
        for fid, alpha in feature_alpha.items()
    }
)

print("=" * 70)


# ============================================================
# OPTIONAL: BASELINE CHECK
# ============================================================

reference_word = "leopard"

reference_margin = intervention_margin(
    reference_word
)

print(
    f"\nReference baseline margin "
    f"({reference_word}): "
    f"{reference_margin:+.4f}"
)

Feature 54316 (animal): mean active SAE activation = 0.7134
Feature 110833 (vehicle): mean active SAE activation = 0.6097

FINAL CAUSAL HELPERS READY
FEATURES: {54316: 'animal', 110833: 'vehicle'}
decoder_directions: [54316, 110833]
feature_alpha: {54316: 0.7134, 110833: 0.6097}

Reference baseline margin (leopard): +12.3750


In [ ]:
# ============================================================
# DIAGNOSTIC — CURRENT CAUSAL STATE
# ============================================================

print("FEATURES =", FEATURES)
print("FEATURE_A =", FEATURE_A)
print("FEATURE_B =", FEATURE_B)

print(
    "decoder_directions =",
    list(decoder_directions.keys())
)

print(
    "feature_alpha =",
    feature_alpha
)

print(
    "54316 in FEATURES:",
    54316 in FEATURES
)

print(
    "54316 in decoder_directions:",
    54316 in decoder_directions
)

print(
    "54316 in feature_alpha:",
    54316 in feature_alpha
)

FEATURES = {54316: 'animal', 110833: 'vehicle'}
FEATURE_A = 54316
FEATURE_B = 110833
decoder_directions = [54316, 110833]
feature_alpha = {54316: 0.7133883833885193, 110833: 0.6096917986869812}
54316 in FEATURES: True
54316 in decoder_directions: True
54316 in feature_alpha: True


## 5.5 Causal Intervention Strength Sweep

We evaluate multiple intervention strengths and both positive and negative
directions.

The central hypothesis is that increasing F54316 should shift the
animal-vs-vehicle margin toward the animal direction.

In [ ]:
# ============================================================
# FINAL CAUSAL SWEEP — RESIDUAL-NORM CALIBRATED
# ============================================================

NORM_FRACTIONS = [
    0.025,
    0.05,
    0.10,
    0.15,
    0.20,
]

final_rows = []

for word, label in final_causal_words:

    baseline = final_baselines[word]

    for fid, fname in FEATURES.items():

        for fraction in NORM_FRACTIONS:

            # --------------------------------------------
            # + direction
            # --------------------------------------------

            strength = fraction * ref_norm

            m_up = intervention_margin(
                word,
                feature_id=fid,
                multiplier=strength / feature_alpha[fid],
                sign=+1.0,
            )

            # --------------------------------------------
            # - direction
            # --------------------------------------------

            m_down = intervention_margin(
                word,
                feature_id=fid,
                multiplier=strength / feature_alpha[fid],
                sign=-1.0,
            )

            increase_delta = (
                m_up - baseline
            )

            suppression_delta = (
                m_down - baseline
            )

            signed_effect = (
                increase_delta
                - suppression_delta
            ) / 2.0

            final_rows.append({
                "word": word,
                "label": label,
                "feature": fid,
                "feature_name": fname,
                "fraction": fraction,
                "strength": strength,
                "baseline": baseline,
                "increase_delta": increase_delta,
                "suppression_delta": suppression_delta,
                "signed_effect": signed_effect,
            })


final_causal_results = pd.DataFrame(
    final_rows
)

print(
    "Final causal results shape:",
    final_causal_results.shape
)

display(
    final_causal_results.head(10)
)

Final causal results shape: (200, 10)


,word,label,feature,feature_name,fraction,strength,baseline,increase_delta,suppression_delta,signed_effect
0,leopard,animal,54316,animal,0.025,0.318338,12.375,0.000,-0.250,0.1250
1,leopard,animal,54316,animal,0.050,0.636677,12.375,-0.125,0.000,-0.0625
2,leopard,animal,54316,animal,0.100,1.273353,12.375,0.125,0.125,0.0000
3,leopard,animal,54316,animal,0.150,1.910030,12.375,0.375,-0.375,0.3750
4,leopard,animal,54316,animal,0.200,2.546707,12.375,0.125,-0.625,0.3750
5,leopard,animal,110833,vehicle,0.025,0.318338,12.375,-0.125,-0.125,0.0000
6,leopard,animal,110833,vehicle,0.050,0.636677,12.375,0.625,0.625,0.0000
7,leopard,animal,110833,vehicle,0.100,1.273353,12.375,0.125,-0.250,0.1875
8,leopard,animal,110833,vehicle,0.150,1.910030,12.375,-0.375,0.125,-0.2500
9,leopard,animal,110833,vehicle,0.200,2.546707,12.375,0.250,1.375,-0.5625


### Causal Result

Increasing F54316 produces a systematic animalward change in the behavioral
margin across the causal-test set.

In [ ]:
# ============================================================
# FINAL CAUSAL SUMMARY
# ============================================================

final_summary = (
    final_causal_results
    .groupby(
        [
            "feature",
            "feature_name",
            "fraction",
        ]
    )
    .agg(
        mean_effect=("signed_effect", "mean"),
        std_effect=("signed_effect", "std"),
        mean_increase=("increase_delta", "mean"),
        mean_suppression=("suppression_delta", "mean"),
        n=("signed_effect", "count"),
    )
    .reset_index()
)

print("=== FINAL CAUSAL SUMMARY ===")
display(final_summary)

=== FINAL CAUSAL SUMMARY ===


,feature,feature_name,fraction,mean_effect,std_effect,mean_increase,mean_suppression,n
0,54316,animal,0.025,0.150000,0.260787,0.268750,-0.03125,20
1,54316,animal,0.050,0.234375,0.311429,0.331250,-0.13750,20
2,54316,animal,0.100,0.321875,0.447616,0.418750,-0.22500,20
3,54316,animal,0.150,0.465625,0.507540,0.643750,-0.28750,20
4,54316,animal,0.200,0.596875,0.528573,0.768750,-0.42500,20
5,110833,vehicle,0.025,0.100000,0.242825,0.256250,0.05625,20
6,110833,vehicle,0.050,-0.065625,0.256073,0.125000,0.25625,20
7,110833,vehicle,0.100,-0.015625,0.351732,0.168750,0.20000,20
8,110833,vehicle,0.150,-0.217188,0.237230,0.021875,0.45625,20
9,110833,vehicle,0.200,-0.135937,0.375175,0.115625,0.38750,20


In [ ]:
# ============================================================
# CATEGORY-SPECIFIC CAUSAL EFFECT
# ============================================================

final_category_summary = (
    final_causal_results
    .groupby(
        [
            "feature",
            "feature_name",
            "fraction",
            "label",
        ]
    )
    .agg(
        mean_effect=("signed_effect", "mean"),
        std_effect=("signed_effect", "std"),
        n=("signed_effect", "count"),
    )
    .reset_index()
)

print("=== CATEGORY-SPECIFIC EFFECT ===")
display(final_category_summary)

=== CATEGORY-SPECIFIC EFFECT ===


,feature,feature_name,fraction,label,mean_effect,std_effect,n
0,54316,animal,0.025,animal,0.137500,0.263194,10
1,54316,animal,0.025,vehicle,0.162500,0.271953,10
2,54316,animal,0.050,animal,0.250000,0.326758,10
3,54316,animal,0.050,vehicle,0.218750,0.312153,10
4,54316,animal,0.100,animal,0.418750,0.438045,10
5,54316,animal,0.100,vehicle,0.225000,0.458523,10
6,54316,animal,0.150,animal,0.606250,0.573617,10
7,54316,animal,0.150,vehicle,0.325000,0.413320,10
8,54316,animal,0.200,animal,0.818750,0.572405,10
9,54316,animal,0.200,vehicle,0.375000,0.390868,10


In [ ]:
# ============================================================
# F54316: DIRECTIONAL CONSISTENCY ON FINAL CAUSAL SET
# ============================================================

fid = 54316
fraction = 0.20

subset = final_causal_results[
    (final_causal_results["feature"] == fid)
    & (final_causal_results["fraction"] == fraction)
].copy()

effects = subset["signed_effect"].to_numpy()

animal_mask = (
    subset["label"] == BEHAVIOR["cat_a"]
).to_numpy()

vehicle_mask = ~animal_mask

animal_effects = effects[animal_mask]
vehicle_effects = effects[vehicle_mask]

print("F54316 @ 0.20 × residual norm")

print(
    f"Animal examples : "
    f"mean={animal_effects.mean():+.4f}, "
    f"positive={np.sum(animal_effects > 0)}/{len(animal_effects)}, "
    f"negative={np.sum(animal_effects < 0)}/{len(animal_effects)}"
)

print(
    f"Vehicle examples: "
    f"mean={vehicle_effects.mean():+.4f}, "
    f"positive={np.sum(vehicle_effects > 0)}/{len(vehicle_effects)}, "
    f"negative={np.sum(vehicle_effects < 0)}/{len(vehicle_effects)}"
)

print(
    f"\nOverall positive-direction fraction: "
    f"{np.mean(effects > 0):.1%}"
)

print(
    f"Animal positive fraction: "
    f"{np.mean(animal_effects > 0):.1%}"
)

print(
    f"Vehicle positive fraction: "
    f"{np.mean(vehicle_effects > 0):.1%}"
)

F54316 @ 0.20 × residual norm
Animal examples : mean=+0.8187, positive=10/10, negative=0/10
Vehicle examples: mean=+0.3750, positive=7/10, negative=1/10

Overall positive-direction fraction: 85.0%
Animal positive fraction: 100.0%
Vehicle positive fraction: 70.0%


# 6. Robustness

We repeat the F54316 intervention using multiple prompt templates while
keeping the underlying words unchanged.


In [ ]:
# ============================================================
# PROMPT VARIANTS FOR ROBUSTNESS TEST
# ============================================================

PROMPT_VARIANTS = {
    "standard": "The {word} is",

    "simple": "{word} is",

    "classification": "Classify the following: {word} is",

    "statement": "This is a {word}. The {word} is",
}

print("=== PROMPT VARIANTS ===")

for name, template in PROMPT_VARIANTS.items():
    print(f"{name:16s}: {template}")

assert len(PROMPT_VARIANTS) >= 3
print("\nPrompt variants ready. ✅")

=== PROMPT VARIANTS ===
standard        : The {word} is
simple          : {word} is
classification  : Classify the following: {word} is
statement       : This is a {word}. The {word} is

Prompt variants ready. ✅


In [ ]:
# ============================================================
# TEST F54316 ACROSS PROMPT FORMS
# ============================================================

FID = 54316

# Use the same final causal-test words.
# We are NOT selecting new words here.

PROMPT_TEST_WORDS = final_causal_words

# Use the strongest calibrated intervention that gave us
# the clearest F54316 result.
TEST_FRACTION = 0.20

prompt_rows = []


def prompt_margin(word, prompt_template):
    prompt = prompt_template.format(word=word)

    lp_a = seq_logprob(
        prompt,
        BEHAVIOR["cont_a"]
    )

    lp_b = seq_logprob(
        prompt,
        BEHAVIOR["cont_b"]
    )

    return lp_a - lp_b


def prompt_intervention_margin(
    word,
    prompt_template,
    fraction,
    sign,
):
    prompt = prompt_template.format(word=word)

    strength = fraction * ref_norm

    lp_a = hooked_seq_logprob(
        prompt,
        BEHAVIOR["cont_a"],
        feature_id=FID,
        strength=strength,
        sign=sign,
    )

    lp_b = hooked_seq_logprob(
        prompt,
        BEHAVIOR["cont_b"],
        feature_id=FID,
        strength=strength,
        sign=sign,
    )

    return lp_a - lp_b


for prompt_name, prompt_template in PROMPT_VARIANTS.items():

    for word, label in PROMPT_TEST_WORDS:

        baseline = prompt_margin(
            word,
            prompt_template
        )

        m_up = prompt_intervention_margin(
            word,
            prompt_template,
            TEST_FRACTION,
            +1.0,
        )

        m_down = prompt_intervention_margin(
            word,
            prompt_template,
            TEST_FRACTION,
            -1.0,
        )

        increase_delta = m_up - baseline
        suppression_delta = m_down - baseline

        signed_effect = (
            increase_delta
            - suppression_delta
        ) / 2.0

        prompt_rows.append({
            "prompt_form": prompt_name,
            "word": word,
            "label": label,
            "baseline": baseline,
            "increase_delta": increase_delta,
            "suppression_delta": suppression_delta,
            "signed_effect": signed_effect,
        })


prompt_robustness = pd.DataFrame(
    prompt_rows
)

print(
    "Prompt robustness shape:",
    prompt_robustness.shape
)

display(
    prompt_robustness.head(12)
)

Prompt robustness shape: (80, 7)


,prompt_form,word,label,baseline,increase_delta,suppression_delta,signed_effect
0,standard,leopard,animal,12.375,0.125,-0.625,0.3750
1,standard,cheetah,animal,11.250,1.000,-0.750,0.8750
2,standard,hyena,animal,12.250,0.250,-0.500,0.3750
3,standard,rhinoceros,animal,15.750,1.750,0.000,0.8750
4,standard,hippopotamus,animal,17.875,1.125,-1.375,1.2500
5,standard,buffalo,animal,13.750,0.250,-1.250,0.7500
6,standard,antelope,animal,12.625,3.000,-1.125,2.0625
7,standard,squirrel,animal,10.750,0.750,-1.500,1.1250
8,standard,flamingo,animal,6.375,0.875,0.000,0.4375
9,standard,meerkat,animal,15.500,-0.625,-0.750,0.0625


In [ ]:
# ============================================================
# PROMPT-FORM SUMMARY
# ============================================================

prompt_summary = (
    prompt_robustness
    .groupby(
        ["prompt_form", "label"]
    )
    .agg(
        baseline_mean=("baseline", "mean"),
        mean_effect=("signed_effect", "mean"),
        std_effect=("signed_effect", "std"),
        positive_fraction=(
            "signed_effect",
            lambda x: (x > 0).mean()
        ),
        n=("signed_effect", "count"),
    )
    .reset_index()
)

print("=== F54316 PROMPT ROBUSTNESS ===")
display(prompt_summary)

=== F54316 PROMPT ROBUSTNESS ===


,prompt_form,label,baseline_mean,mean_effect,std_effect,positive_fraction,n
0,classification,animal,11.20000,0.55000,0.575694,0.9,10
1,classification,vehicle,-4.66250,0.46875,0.774625,0.8,10
2,simple,animal,11.18750,0.32500,0.811591,0.7,10
3,simple,vehicle,-6.07500,0.71875,0.593841,1.0,10
4,standard,animal,12.85000,0.81875,0.572405,1.0,10
5,standard,vehicle,-5.48750,0.37500,0.390868,0.7,10
6,statement,animal,12.72500,0.15625,0.655671,0.7,10
7,statement,vehicle,-7.91875,0.15625,0.468866,0.6,10


### Prompt-Form Robustness Result

The F54316 intervention remains behaviorally active across multiple prompt
formulations, indicating that the observed effect is not restricted to a
single surface form.

The magnitude of the effect varies across prompt types, with the strongest
effect observed for the standard formulation and weaker effects for the
simple and statement formulations.

Thus, the result supports **prompt-form robustness**, but not complete
magnitude invariance. The intervention appears to track the underlying
animal-related behavior across several linguistic contexts.

# 7. Semantic Characterization of F54316

We independently characterize F54316 using fresh semantic probes.

The purpose is to determine what kinds of concepts systematically activate
the feature beyond the original animal-vs-vehicle dataset.

## 7.1 Broad Semantic Profile

F54316 is tested across multiple semantic categories, including animals,
vehicles, food, abstract concepts, technology, tools, people, and places.

## 7.2 Biological Subcategory Profile

We then examine more specific biological categories such as mammals, birds,
reptiles, insects, fish, fungi, plants, and body parts.

In [ ]:
# ============================================================
# FRESH SEMANTIC PROBES — AUTOMATICALLY FILTER ALL USED WORDS
# ============================================================

SEMANTIC_PROBES = {
    "animals": [
        "panther", "jaguar", "chimpanzee", "lemur", "beaver",
        "raccoon", "mongoose", "wombat", "sloth", "armadillo",
    ],

    "vehicles": [
        "submarine", "excavator", "zeppelin", "monorail", "rickshaw",
        "limousine", "hovercraft", "forklift", "snowmobile", "gondola",
    ],

    "food": [
        "strawberry", "watermelon", "mango", "pineapple", "lettuce",
        "broccoli", "chocolate", "pancake", "omelet", "sausage",
    ],

    "people": [
        "doctor", "lawyer", "pilot", "teacher", "artist",
        "engineer", "farmer", "nurse", "chef", "actor",
    ],

    "places": [
        "hospital", "school", "airport", "museum", "library",
        "stadium", "restaurant", "village", "mountain", "island",
    ],

    "tools": [
        "hammer", "screwdriver", "wrench", "pliers", "drill",
        "shovel", "scissors", "chisel", "rake", "crowbar",
    ],

    "clothing": [
        "shirt", "jacket", "trousers", "dress", "coat",
        "hat", "glove", "scarf", "shoe", "sweater",
    ],

    "nature": [
        "tree", "forest", "river", "cloud", "rain",
        "flower", "ocean", "desert", "thunder", "volcano",
    ],

    "technology": [
        "keyboard", "monitor", "processor", "software", "database",
        "server", "router", "smartphone", "tablet", "console",
    ],

    "abstract": [
        "justice", "freedom", "knowledge", "beauty", "truth",
        "memory", "language", "success", "failure", "democracy",
    ],
}

# ------------------------------------------------------------
# Everything used previously in this project
# ------------------------------------------------------------

already_used = (
    set(BEHAVIOR["discovery_a"])
    | set(BEHAVIOR["discovery_b"])
    | set(BEHAVIOR["heldout_a"])
    | set(BEHAVIOR["heldout_b"])
)

if "CAUSAL_TEST" in globals():
    already_used |= (
        set(CAUSAL_TEST["animal"])
        | set(CAUSAL_TEST["vehicle"])
    )

if "FINAL_CAUSAL_TEST" in globals():
    already_used |= (
        set(FINAL_CAUSAL_TEST["animal"])
        | set(FINAL_CAUSAL_TEST["vehicle"])
    )

# ------------------------------------------------------------
# Remove every accidental overlap automatically
# ------------------------------------------------------------

clean_probes = {}

for category, words in SEMANTIC_PROBES.items():

    clean_words = [
        w for w in words
        if w not in already_used
    ]

    clean_probes[category] = clean_words

# Replace original dictionary
SEMANTIC_PROBES = clean_probes

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=== CLEAN SEMANTIC PROBE SET ===")

total = 0

for category, words in SEMANTIC_PROBES.items():

    print(
        f"{category:12s}: "
        f"{len(words)} words"
    )

    if words:
        print("   ", words)

    total += len(words)

print(
    f"\nTotal clean probe words: {total}"
)

# ------------------------------------------------------------
# Final safety check
# ------------------------------------------------------------

probe_words = {
    word
    for words in SEMANTIC_PROBES.values()
    for word in words
}

overlap = already_used & probe_words

print(
    "\nOverlap with previous experiments:",
    overlap
)

assert not overlap

print(
    "Semantic probe set is clean. ✅"
)

=== CLEAN SEMANTIC PROBE SET ===
animals     : 10 words
    ['panther', 'jaguar', 'chimpanzee', 'lemur', 'beaver', 'raccoon', 'mongoose', 'wombat', 'sloth', 'armadillo']
vehicles    : 9 words
    ['submarine', 'excavator', 'zeppelin', 'monorail', 'rickshaw', 'hovercraft', 'forklift', 'snowmobile', 'gondola']
food        : 10 words
    ['strawberry', 'watermelon', 'mango', 'pineapple', 'lettuce', 'broccoli', 'chocolate', 'pancake', 'omelet', 'sausage']
people      : 10 words
    ['doctor', 'lawyer', 'pilot', 'teacher', 'artist', 'engineer', 'farmer', 'nurse', 'chef', 'actor']
places      : 10 words
    ['hospital', 'school', 'airport', 'museum', 'library', 'stadium', 'restaurant', 'village', 'mountain', 'island']
tools       : 10 words
    ['hammer', 'screwdriver', 'wrench', 'pliers', 'drill', 'shovel', 'scissors', 'chisel', 'rake', 'crowbar']
clothing    : 10 words
    ['shirt', 'jacket', 'trousers', 'dress', 'coat', 'hat', 'glove', 'scarf', 'shoe', 'sweater']
nature      : 10 words
  

In [ ]:
# ============================================================
# EXTRACT F54316 ACTIVATIONS
# ============================================================

PROFILE_FEATURE = 54316

profile_rows = []

for category, words in SEMANTIC_PROBES.items():

    for word in words:

        x = get_layer19_activation(word)

        with torch.no_grad():
            sae_features = sae.encode(x).squeeze(0)

        activation = sae_features[
            PROFILE_FEATURE
        ].item()

        profile_rows.append({
            "word": word,
            "category": category,
            "activation": activation,
            "active": activation > 0,
        })

profile_df = pd.DataFrame(profile_rows)

print(
    "Profile shape:",
    profile_df.shape
)

display(profile_df)

Profile shape: (99, 4)


,word,category,activation,active
0,panther,animals,0.906705,True
1,jaguar,animals,1.239790,True
2,chimpanzee,animals,1.434816,True
3,lemur,animals,1.423679,True
4,beaver,animals,1.153611,True
...,...,...,...,...
94,memory,abstract,0.000000,False
95,language,abstract,0.000000,False
96,success,abstract,0.000000,False
97,failure,abstract,0.000000,False


In [ ]:
# ============================================================
# F54316 SEMANTIC PROFILE — CATEGORY SUMMARY
# ============================================================

profile_summary = (
    profile_df
    .groupby("category")
    .agg(
        mean_activation=("activation", "mean"),
        std_activation=("activation", "std"),
        median_activation=("activation", "median"),
        active_fraction=("active", "mean"),
        max_activation=("activation", "max"),
        n=("activation", "count"),
    )
    .sort_values(
        "mean_activation",
        ascending=False
    )
    .reset_index()
)

print("=== F54316 CATEGORY PROFILE ===")
display(profile_summary)

=== F54316 CATEGORY PROFILE ===


,category,mean_activation,std_activation,median_activation,active_fraction,max_activation,n
0,animals,1.307821,0.172279,1.344579,1.000000,1.458555,10
1,vehicles,0.050996,0.152989,0.000000,0.111111,0.458967,9
2,food,0.046058,0.145650,0.000000,0.100000,0.460585,10
3,abstract,0.000000,0.000000,0.000000,0.000000,0.000000,10
4,nature,0.000000,0.000000,0.000000,0.000000,0.000000,10
5,clothing,0.000000,0.000000,0.000000,0.000000,0.000000,10
6,people,0.000000,0.000000,0.000000,0.000000,0.000000,10
7,places,0.000000,0.000000,0.000000,0.000000,0.000000,10
8,technology,0.000000,0.000000,0.000000,0.000000,0.000000,10
9,tools,0.000000,0.000000,0.000000,0.000000,0.000000,10


In [ ]:
# ============================================================
# F54316 — TOP AND BOTTOM EXAMPLES
# ============================================================

print("=== TOP 20 F54316 ACTIVATIONS ===")

top_profile = (
    profile_df
    .sort_values(
        "activation",
        ascending=False
    )
    .head(20)
)

display(top_profile)


print("\n=== BOTTOM 20 F54316 ACTIVATIONS ===")

bottom_profile = (
    profile_df
    .sort_values(
        "activation",
        ascending=True
    )
    .head(20)
)

display(bottom_profile)

=== TOP 20 F54316 ACTIVATIONS ===


,word,category,activation,active
9,armadillo,animals,1.458555,True
6,mongoose,animals,1.457631,True
2,chimpanzee,animals,1.434816,True
3,lemur,animals,1.423679,True
5,raccoon,animals,1.357982,True
7,wombat,animals,1.331175,True
8,sloth,animals,1.314263,True
1,jaguar,animals,1.239790,True
4,beaver,animals,1.153611,True
0,panther,animals,0.906705,True



=== BOTTOM 20 F54316 ACTIVATIONS ===


,word,category,activation,active
13,monorail,vehicles,0.0,False
14,rickshaw,vehicles,0.0,False
10,submarine,vehicles,0.0,False
15,hovercraft,vehicles,0.0,False
11,excavator,vehicles,0.0,False
26,pancake,food,0.0,False
25,chocolate,food,0.0,False
28,sausage,food,0.0,False
24,broccoli,food,0.0,False
23,lettuce,food,0.0,False


In [ ]:
# ============================================================
# F54316 — TARGETED SEMANTIC DISCRIMINATION
# ============================================================

TARGETED_PROBES = {
    "mammals": [
        "fox", "badger", "hyrax", "tapir", "manatee",
        "weasel", "marmot", "ibex",
    ],

    "birds": [
        "sparrow", "falcon", "robin", "swallow",
        "pelican", "heron", "woodpecker", "ostrich",
    ],

    "reptiles": [
        "iguana", "chameleon", "gecko", "crocodile",
        "tortoise", "lizard", "alligator", "python",
    ],

    "fish": [
        "salmon", "trout", "tuna", "cod",
        "sardine", "mackerel", "carp", "anchovy",
    ],

    "insects": [
        "beetle", "butterfly", "dragonfly", "mosquito",
        "termite", "ladybug", "grasshopper", "ant",
    ],

    "plants": [
        "oak", "pine", "maple", "bamboo",
        "fern", "cactus", "moss", "willow",
    ],

    "fungi": [
        "mushroom", "truffle", "yeast", "mildew",
        "lichen", "morel",
    ],

    "body_parts": [
        "heart", "lung", "brain", "kidney",
        "stomach", "bone", "muscle", "finger",
    ],
}


# ------------------------------------------------------------
# AUTOMATIC LEAKAGE FILTER
# ------------------------------------------------------------

already_used = (
    set(BEHAVIOR["discovery_a"])
    | set(BEHAVIOR["discovery_b"])
    | set(BEHAVIOR["heldout_a"])
    | set(BEHAVIOR["heldout_b"])
)

if "CAUSAL_TEST" in globals():
    already_used |= (
        set(CAUSAL_TEST["animal"])
        | set(CAUSAL_TEST["vehicle"])
    )

if "FINAL_CAUSAL_TEST" in globals():
    already_used |= (
        set(FINAL_CAUSAL_TEST["animal"])
        | set(FINAL_CAUSAL_TEST["vehicle"])
    )

if "SEMANTIC_PROBES" in globals():
    already_used |= {
        w
        for words in SEMANTIC_PROBES.values()
        for w in words
    }


# ------------------------------------------------------------
# REMOVE ANY ACCIDENTAL OVERLAP
# ------------------------------------------------------------

clean_targeted = {}

for category, words in TARGETED_PROBES.items():

    clean_words = [
        w for w in words
        if w not in already_used
    ]

    clean_targeted[category] = clean_words

TARGETED_PROBES = clean_targeted


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=== CLEAN TARGETED PROBES ===")

for category, words in TARGETED_PROBES.items():
    print(
        f"{category:12s}: {len(words)} words"
    )
    print("   ", words)


all_targeted = {
    w
    for words in TARGETED_PROBES.values()
    for w in words
}

assert not (already_used & all_targeted)

print(
    "\nTargeted probe set is clean. ✅"
)

=== CLEAN TARGETED PROBES ===
mammals     : 6 words
    ['hyrax', 'tapir', 'manatee', 'weasel', 'marmot', 'ibex']
birds       : 8 words
    ['sparrow', 'falcon', 'robin', 'swallow', 'pelican', 'heron', 'woodpecker', 'ostrich']
reptiles    : 8 words
    ['iguana', 'chameleon', 'gecko', 'crocodile', 'tortoise', 'lizard', 'alligator', 'python']
fish        : 8 words
    ['salmon', 'trout', 'tuna', 'cod', 'sardine', 'mackerel', 'carp', 'anchovy']
insects     : 8 words
    ['beetle', 'butterfly', 'dragonfly', 'mosquito', 'termite', 'ladybug', 'grasshopper', 'ant']
plants      : 8 words
    ['oak', 'pine', 'maple', 'bamboo', 'fern', 'cactus', 'moss', 'willow']
fungi       : 6 words
    ['mushroom', 'truffle', 'yeast', 'mildew', 'lichen', 'morel']
body_parts  : 8 words
    ['heart', 'lung', 'brain', 'kidney', 'stomach', 'bone', 'muscle', 'finger']

Targeted probe set is clean. ✅


In [ ]:
# ============================================================
# EXTRACT F54316 ON TARGETED PROBES
# ============================================================

targeted_rows = []

for category, words in TARGETED_PROBES.items():

    for word in words:

        x = get_layer19_activation(word)

        with torch.no_grad():
            feats = sae.encode(x).squeeze(0)

        activation = feats[54316].item()

        targeted_rows.append({
            "word": word,
            "category": category,
            "activation": activation,
            "active": activation > 0,
        })


targeted_df = pd.DataFrame(targeted_rows)

print(
    "Targeted profile shape:",
    targeted_df.shape
)

display(targeted_df)

Targeted profile shape: (60, 4)


,word,category,activation,active
0,hyrax,mammals,1.664640,True
1,tapir,mammals,1.456952,True
2,manatee,mammals,1.671739,True
3,weasel,mammals,1.403491,True
4,marmot,mammals,1.361912,True
5,ibex,mammals,1.351928,True
6,sparrow,birds,1.208150,True
7,falcon,birds,0.462826,True
8,robin,birds,1.082946,True
9,swallow,birds,0.718548,True


In [ ]:
# ============================================================
# TARGETED SUMMARY
# ============================================================

targeted_summary = (
    targeted_df
    .groupby("category")
    .agg(
        mean_activation=("activation", "mean"),
        std_activation=("activation", "std"),
        active_fraction=("active", "mean"),
        max_activation=("activation", "max"),
        n=("activation", "count"),
    )
    .sort_values(
        "mean_activation",
        ascending=False
    )
    .reset_index()
)

print("=== F54316 TARGETED SEMANTIC PROFILE ===")
display(targeted_summary)

=== F54316 TARGETED SEMANTIC PROFILE ===


,category,mean_activation,std_activation,active_fraction,max_activation,n
0,mammals,1.485110,0.146582,1.000000,1.671739,6
1,birds,1.098328,0.338818,1.000000,1.426442,8
2,reptiles,0.931567,0.484697,0.875000,1.384402,8
3,insects,0.704687,0.371635,0.875000,1.122970,8
4,fish,0.261833,0.286904,0.500000,0.640416,8
5,fungi,0.187932,0.293872,0.333333,0.626977,6
6,plants,0.141269,0.261580,0.250000,0.566419,8
7,body_parts,0.000000,0.000000,0.000000,0.000000,8


## 7.3 Semantic Interpretation

F54316 shows its strongest activation for animal-related concepts, with
particularly strong activation for mammals and substantial activation across
birds, reptiles, and insects.

We therefore characterize F54316 as an **animal-related SAE feature**.

This is an empirical characterization of the feature's activation profile,
not a claim that the feature has a single complete semantic definition.

## 7.4 Contrastive Semantic Test

A paired comparison between animal examples and non-animal controls provides
an additional test of whether F54316 preferentially activates for
animal-related content.

In [ ]:
# ============================================================
# CONTRASTIVE FEATURE ATTRIBUTION (CFA)
# ============================================================

CFA_PAIRS = [
    ("panther", "submarine"),
    ("chimpanzee", "database"),
    ("lemur", "hammer"),
    ("beaver", "hospital"),
    ("raccoon", "keyboard"),
    ("mongoose", "jacket"),
    ("wombat", "democracy"),
    ("jaguar", "pancake"),
    ("armadillo", "volcano"),
    ("sloth", "screwdriver"),
    ("sparrow", "server"),
    ("falcon", "restaurant"),
    ("iguana", "camera"),
    ("crocodile", "library"),
    ("beetle", "tractor"),
]

In [ ]:
# ============================================================
# CONTRASTIVE FEATURE ATTRIBUTION — FRESH WORDS ONLY
# ============================================================

import random

# ------------------------------------------------------------
# Candidate vocabulary, intentionally larger than needed
# ------------------------------------------------------------

CFA_CANDIDATES = {
    "animal": [
        "gazelle", "moose", "reindeer", "yak", "llama",
        "alpaca", "chimpanzee", "baboon", "gibbon", "orangutan",
        "lemur", "hyrax", "pangolin", "aardvark", "porcupine",
        "hamster", "gerbil", "salamander", "newt", "salamander",
        "pelican", "crane", "magpie", "raven", "parrot",
        "owl", "vulture", "tern", "pigeon", "kingfisher",
    ],

    "control": [
        "notebook", "curtain", "pillow", "mirror", "drawer",
        "carpet", "bucket", "basket", "umbrella", "lantern",
        "stapler", "envelope", "wallet", "necklace", "calendar",
        "window", "blanket", "cushion", "broom", "suitcase",
        "mug", "plate", "fork", "spoon", "napkin",
        "helmet", "sandal", "pencil", "eraser", "ruler",
    ],
}

# ------------------------------------------------------------
# Collect EVERYTHING already used
# ------------------------------------------------------------

already_used = set()

# Original behavior sets
already_used |= (
    set(BEHAVIOR["discovery_a"])
    | set(BEHAVIOR["discovery_b"])
    | set(BEHAVIOR["heldout_a"])
    | set(BEHAVIOR["heldout_b"])
)

# Previous causal sets
if "CAUSAL_TEST" in globals():
    already_used |= (
        set(CAUSAL_TEST["animal"])
        | set(CAUSAL_TEST["vehicle"])
    )

if "FINAL_CAUSAL_TEST" in globals():
    already_used |= (
        set(FINAL_CAUSAL_TEST["animal"])
        | set(FINAL_CAUSAL_TEST["vehicle"])
    )

# Previous semantic profiling probes
if "SEMANTIC_PROBES" in globals():
    already_used |= {
        w
        for words in SEMANTIC_PROBES.values()
        for w in words
    }

if "TARGETED_PROBES" in globals():
    already_used |= {
        w
        for words in TARGETED_PROBES.values()
        for w in words
    }

# ------------------------------------------------------------
# Automatically remove every previously used word
# ------------------------------------------------------------

fresh_animals = [
    w for w in CFA_CANDIDATES["animal"]
    if w not in already_used
]

fresh_controls = [
    w for w in CFA_CANDIDATES["control"]
    if w not in already_used
]

# Remove duplicates
fresh_animals = list(dict.fromkeys(fresh_animals))
fresh_controls = list(dict.fromkeys(fresh_controls))

# Need equal numbers
N_PAIRS = min(
    10,
    len(fresh_animals),
    len(fresh_controls)
)

assert N_PAIRS >= 10, (
    f"Not enough fresh words. "
    f"Animals={len(fresh_animals)}, "
    f"Controls={len(fresh_controls)}"
)

# Deterministic selection
random.seed(54316)

random.shuffle(fresh_animals)
random.shuffle(fresh_controls)

fresh_animals = fresh_animals[:N_PAIRS]
fresh_controls = fresh_controls[:N_PAIRS]

CFA_PAIRS = list(
    zip(fresh_animals, fresh_controls)
)

# ------------------------------------------------------------
# Final leakage check
# ------------------------------------------------------------

cfa_words = {
    w
    for pair in CFA_PAIRS
    for w in pair
}

overlap = already_used & cfa_words

print("=== FRESH CFA PAIRS ===")

for i, (animal, control) in enumerate(CFA_PAIRS, 1):
    print(
        f"{i:2d}. {animal:15s} ↔ {control}"
    )

print(
    "\nOverlap with previous experiments:",
    overlap
)

assert not overlap, (
    f"CFA leakage detected: {overlap}"
)

print(
    "\nCFA word set is clean. ✅"
)

=== FRESH CFA PAIRS ===
 1. parrot          ↔ calendar
 2. owl             ↔ eraser
 3. vulture         ↔ plate
 4. crane           ↔ napkin
 5. magpie          ↔ carpet
 6. orangutan       ↔ notebook
 7. kingfisher      ↔ cushion
 8. hamster         ↔ spoon
 9. llama           ↔ pencil
10. salamander      ↔ mirror

Overlap with previous experiments: set()

CFA word set is clean. ✅


In [ ]:
# ============================================================
# CFA — F54316 ACTIVATION DIFFERENCE
# ============================================================

CFA_FEATURE = 54316

cfa_rows = []

for animal_word, control_word in CFA_PAIRS:

    x_animal = get_layer19_activation(animal_word)
    x_control = get_layer19_activation(control_word)

    with torch.no_grad():
        z_animal = sae.encode(x_animal).squeeze(0)
        z_control = sae.encode(x_control).squeeze(0)

    a = z_animal[CFA_FEATURE].item()
    c = z_control[CFA_FEATURE].item()

    cfa_rows.append({
        "animal_word": animal_word,
        "control_word": control_word,
        "animal_activation": a,
        "control_activation": c,
        "contrast": a - c,
        "animal_higher": a > c,
    })

cfa_df = pd.DataFrame(cfa_rows)

display(cfa_df)

,animal_word,control_word,animal_activation,control_activation,contrast,animal_higher
0,parrot,calendar,0.000000,0.0,0.000000,False
1,owl,eraser,0.713277,0.0,0.713277,True
2,vulture,plate,0.896401,0.0,0.896401,True
3,crane,napkin,0.667623,0.0,0.667623,True
4,magpie,carpet,0.836181,0.0,0.836181,True
5,orangutan,notebook,1.584963,0.0,1.584963,True
6,kingfisher,cushion,1.421328,0.0,1.421328,True
7,hamster,spoon,0.579291,0.0,0.579291,True
8,llama,pencil,1.117543,0.0,1.117543,True
9,salamander,mirror,1.261625,0.0,1.261625,True


In [ ]:
# ============================================================
# CFA SUMMARY
# ============================================================

print("=== F54316 CONTRASTIVE FEATURE ATTRIBUTION ===")

print(
    f"Mean contrast: "
    f"{cfa_df['contrast'].mean():+.4f}"
)

print(
    f"Animal activation higher: "
    f"{cfa_df['animal_higher'].mean():.1%}"
)

print(
    f"Pairs: "
    f"{int(cfa_df['animal_higher'].sum())}/"
    f"{len(cfa_df)}"
)

=== F54316 CONTRASTIVE FEATURE ATTRIBUTION ===
Mean contrast: +0.9078
Animal activation higher: 90.0%
Pairs: 9/10


# 8. Circuit Tracing

After establishing a causal behavioral role for F54316, we investigate where
the feature sits within the model's computation.

We examine:

1. upstream layers,
2. attention and MLP components,
3. individual attention heads,
4. feature-mediated rescue,
5. downstream mediation, and
6. downstream propagation.

## 8.1 Upstream Layer Analysis

We ablate earlier transformer layers and measure their effect on F54316.

This identifies layers that contribute to the formation of the feature.

In [ ]:
# ============================================================
# CIRCUIT TRACING — STEP 1
# LAYER-LEVEL UPSTREAM SCAN FOR F54316
# ============================================================

FEATURE_ID = 54316

# Use fresh examples so this is not tied to one word.
TRACE_WORDS = [
    "panther",
    "jaguar",
    "chimpanzee",
    "lemur",
    "beaver",
    "raccoon",
    "wombat",
    "sloth",
]

print("Tracing feature:", FEATURE_ID)
print("Tracing words:", TRACE_WORDS)

Tracing feature: 54316
Tracing words: ['panther', 'jaguar', 'chimpanzee', 'lemur', 'beaver', 'raccoon', 'wombat', 'sloth']


In [ ]:
# ============================================================
# BASELINE F54316 ACTIVATIONS
# ============================================================

baseline_feature_acts = {}

for word in TRACE_WORDS:

    x = get_layer19_activation(word)

    with torch.no_grad():
        z = sae.encode(x).squeeze(0)

    baseline_feature_acts[word] = z[
        FEATURE_ID
    ].item()

    print(
        f"{word:12s} "
        f"F{FEATURE_ID} = "
        f"{baseline_feature_acts[word]:.4f}"
    )

panther      F54316 = 0.9067
jaguar       F54316 = 1.2398
chimpanzee   F54316 = 1.4348
lemur        F54316 = 1.4237
beaver       F54316 = 1.1536
raccoon      F54316 = 1.3580
wombat       F54316 = 1.3312
sloth        F54316 = 1.3143


In [ ]:
# ============================================================
# INSPECT LLAMA LAYER STRUCTURE
# ============================================================

layer_idx = 10

layer = model.model.layers[layer_idx]

print(layer)

print("\nSubmodules:")

for name, module in layer.named_children():
    print(
        f"{name:30s}",
        type(module).__name__
    )

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear8bitLt(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear8bitLt(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear8bitLt(in_features=4096, out_features=14336, bias=False)
    (up_proj): Linear8bitLt(in_features=4096, out_features=14336, bias=False)
    (down_proj): Linear8bitLt(in_features=14336, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)

Submodules:
self_attn                      LlamaAttention
mlp                            LlamaMLP
input_layernorm                LlamaRMSNorm
post_attention_layernorm       LlamaRMSNorm


In [ ]:
# ============================================================
# CIRCUIT TRACING — UPSTREAM LAYER ABLATION
# ============================================================

FEATURE_ID = 54316


def get_layer19_feature_with_layer_ablation(
    word,
    ablate_layer=None,
):
    """
    Run the normal model, optionally ablating the entire
    contribution of one earlier transformer block at the
    final-token position.

    Ablation:
        layer_output[last] <- layer_input[last]

    This removes the block's residual update at that position.
    """

    captured = {}

    # --------------------------------------------------------
    # Hook Layer 19 to capture the SAE input
    # --------------------------------------------------------

    def layer19_hook(module, inputs, output):
        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )
        captured["layer19"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    h19 = model.model.layers[19].register_forward_hook(
        layer19_hook
    )

    # --------------------------------------------------------
    # Optional upstream layer ablation
    # --------------------------------------------------------

    hablate = None

    if ablate_layer is not None:

        if not (0 <= ablate_layer < 19):
            raise ValueError(
                "ablate_layer must be in [0, 18]"
            )

        def ablation_hook(module, inputs, output):

            hidden_in = inputs[0]

            if isinstance(output, tuple):
                hidden_out = output[0].clone()

                # Remove this layer's residual update
                hidden_out[:, -1, :] = hidden_in[:, -1, :]

                return (
                    (hidden_out,)
                    + output[1:]
                )

            else:
                hidden_out = output.clone()

                hidden_out[:, -1, :] = hidden_in[:, -1, :]

                return hidden_out

        hablate = model.model.layers[
            ablate_layer
        ].register_forward_hook(
            ablation_hook
        )

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    try:

        messages = [
            {
                "role": "user",
                "content": make_prompt(word),
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():

            _ = model(
                **inputs,
                use_cache=False,
            )

    finally:

        h19.remove()

        if hablate is not None:
            hablate.remove()

    # --------------------------------------------------------
    # Encode Layer-19 residual through SAE
    # --------------------------------------------------------

    x = captured["layer19"]

    with torch.no_grad():
        z = sae.encode(x)

    return z[0, FEATURE_ID].item()

In [ ]:
# ============================================================
# UPSTREAM LAYER SCREEN
# ============================================================

TRACE_WORDS = [
    "panther",
    "jaguar",
    "chimpanzee",
    "lemur",
    "beaver",
    "raccoon",
    "wombat",
    "sloth",
]

# ------------------------------------------------------------
# Baselines
# ------------------------------------------------------------

baseline_trace = {}

print("=== BASELINE F54316 ===")

for word in TRACE_WORDS:

    value = get_layer19_feature_with_layer_ablation(
        word,
        ablate_layer=None,
    )

    baseline_trace[word] = value

    print(
        f"{word:12s} "
        f"F{FEATURE_ID}={value:.4f}"
    )


# ------------------------------------------------------------
# Ablate each upstream layer
# ------------------------------------------------------------

layer_rows = []

for layer_idx in range(19):

    effects = []

    print(
        f"\nTesting Layer {layer_idx}..."
    )

    for word in TRACE_WORDS:

        baseline = baseline_trace[word]

        ablated = (
            get_layer19_feature_with_layer_ablation(
                word,
                ablate_layer=layer_idx,
            )
        )

        delta = ablated - baseline

        effects.append(delta)

        layer_rows.append({
            "layer": layer_idx,
            "word": word,
            "baseline": baseline,
            "ablated": ablated,
            "delta": delta,
            "abs_delta": abs(delta),
        })

    print(
        f"Layer {layer_idx:2d} | "
        f"mean ΔF54316 = {np.mean(effects):+.4f} | "
        f"mean |Δ| = {np.mean(np.abs(effects)):.4f}"
    )


layer_ablation_df = pd.DataFrame(
    layer_rows
)

print(
    "\nLayer ablation shape:",
    layer_ablation_df.shape
)

=== BASELINE F54316 ===
panther      F54316=0.9067
jaguar       F54316=1.2398
chimpanzee   F54316=1.4348
lemur        F54316=1.4237
beaver       F54316=1.1536
raccoon      F54316=1.3580
wombat       F54316=1.3312
sloth        F54316=1.3143

Testing Layer 0...
Layer  0 | mean ΔF54316 = -1.0755 | mean |Δ| = 1.0755

Testing Layer 1...
Layer  1 | mean ΔF54316 = +0.0235 | mean |Δ| = 0.0426

Testing Layer 2...
Layer  2 | mean ΔF54316 = +0.0764 | mean |Δ| = 0.0807

Testing Layer 3...
Layer  3 | mean ΔF54316 = -0.1562 | mean |Δ| = 0.1562

Testing Layer 4...
Layer  4 | mean ΔF54316 = -0.0152 | mean |Δ| = 0.0279

Testing Layer 5...
Layer  5 | mean ΔF54316 = -0.0296 | mean |Δ| = 0.0646

Testing Layer 6...
Layer  6 | mean ΔF54316 = -0.3418 | mean |Δ| = 0.3418

Testing Layer 7...
Layer  7 | mean ΔF54316 = -0.2104 | mean |Δ| = 0.2104

Testing Layer 8...
Layer  8 | mean ΔF54316 = -0.3238 | mean |Δ| = 0.3238

Testing Layer 9...
Layer  9 | mean ΔF54316 = -0.2575 | mean |Δ| = 0.2575

Testing Layer 10...

In [ ]:
# ============================================================
# UPSTREAM LAYER RANKING
# ============================================================

layer_summary = (
    layer_ablation_df
    .groupby("layer")
    .agg(
        mean_delta=("delta", "mean"),
        mean_abs_delta=("abs_delta", "mean"),
        std_delta=("delta", "std"),
        n=("delta", "count"),
    )
    .sort_values(
        "mean_abs_delta",
        ascending=False
    )
    .reset_index()
)

print("=== UPSTREAM LAYER RANKING ===")
display(layer_summary)

=== UPSTREAM LAYER RANKING ===


,layer,mean_delta,mean_abs_delta,std_delta,n
0,0,-1.075468,1.075468,0.222145,8
1,17,-0.655406,0.655406,0.081292,8
2,18,-0.492868,0.492868,0.080428,8
3,6,-0.341829,0.341829,0.086440,8
4,8,-0.323823,0.323823,0.076899,8
5,15,-0.257575,0.257575,0.058125,8
6,9,-0.257490,0.257490,0.052043,8
7,11,0.212443,0.212443,0.080340,8
8,7,-0.210381,0.210381,0.094670,8
9,14,-0.192430,0.192430,0.050114,8


### Upstream Layer Screening Result

Layer 0 shows the largest absolute change in F54316 under layer ablation,
followed by Layers 17 and 18.

Because very early-layer ablation can produce broad downstream disruption, we
do not interpret the magnitude of the Layer-0 effect alone as evidence of a
specific F54316-producing mechanism.

We therefore proceed to component-level analysis of Layer 17,18 and a few as a candidate late-stage contributor to F54316 formation.

## 8.2 Identifying the Upstream Component

After identifying the strongest upstream layers, we separate attention and
MLP contributions to determine which component most strongly affects F54316.

In [ ]:
# ============================================================
# CIRCUIT TRACING — ATTENTION vs MLP
# ============================================================

FEATURE_ID = 54316

CANDIDATE_LAYERS = [
    17,
    18,
    6,
    8,
    15,
    9,
    11,
]

def get_feature_with_component_ablation(
    word,
    layer_idx,
    component,
):
    """
    Ablate either the self-attention or MLP residual update
    at the final token position of one transformer layer.
    """

    captured = {}

    # --------------------------------------------------------
    # Capture Layer-19 residual
    # --------------------------------------------------------

    def layer19_hook(module, inputs, output):
        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        captured["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    h19 = model.model.layers[19].register_forward_hook(
        layer19_hook
    )

    # --------------------------------------------------------
    # Ablate selected component
    # --------------------------------------------------------

    if component == "attn":
        target_module = (
            model.model.layers[layer_idx].self_attn
        )

    elif component == "mlp":
        target_module = (
            model.model.layers[layer_idx].mlp
        )

    else:
        raise ValueError(
            "component must be 'attn' or 'mlp'"
        )

    def component_hook(module, inputs, output):

        # Attention and MLP both return a tensor in this
        # Transformers implementation, but handle tuples safely.
        if isinstance(output, tuple):

            hidden = output[0].clone()

            # Remove component contribution at final token.
            hidden[:, -1, :] = 0.0

            return (
                (hidden,)
                + output[1:]
            )

        else:

            hidden = output.clone()

            hidden[:, -1, :] = 0.0

            return hidden

    hc = target_module.register_forward_hook(
        component_hook
    )

    try:

        messages = [
            {
                "role": "user",
                "content": make_prompt(word),
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():

            _ = model(
                **inputs,
                use_cache=False,
            )

    finally:

        hc.remove()
        h19.remove()

    with torch.no_grad():

        z = sae.encode(
            captured["x"]
        )

    return z[0, FEATURE_ID].item()

In [ ]:
# ============================================================
# COMPONENT SCREEN
# ============================================================

component_rows = []

for layer_idx in CANDIDATE_LAYERS:

    print(f"\n=== LAYER {layer_idx} ===")

    for component in ["attn", "mlp"]:

        effects = []

        for word in TRACE_WORDS:

            baseline = baseline_trace[word]

            ablated = get_feature_with_component_ablation(
                word,
                layer_idx,
                component,
            )

            delta = ablated - baseline

            effects.append(delta)

            component_rows.append({
                "layer": layer_idx,
                "component": component,
                "word": word,
                "baseline": baseline,
                "ablated": ablated,
                "delta": delta,
                "abs_delta": abs(delta),
            })

        print(
            f"{component:5s} | "
            f"mean ΔF54316 = {np.mean(effects):+.4f} | "
            f"mean |Δ| = {np.mean(np.abs(effects)):.4f}"
        )


component_df = pd.DataFrame(
    component_rows
)

print(
    "\nComponent result shape:",
    component_df.shape
)


=== LAYER 17 ===
attn  | mean ΔF54316 = -0.2185 | mean |Δ| = 0.2185
mlp   | mean ΔF54316 = -0.3662 | mean |Δ| = 0.3662

=== LAYER 18 ===
attn  | mean ΔF54316 = -0.4548 | mean |Δ| = 0.4548
mlp   | mean ΔF54316 = +0.0673 | mean |Δ| = 0.0673

=== LAYER 6 ===
attn  | mean ΔF54316 = -0.1908 | mean |Δ| = 0.1908
mlp   | mean ΔF54316 = -0.2374 | mean |Δ| = 0.2374

=== LAYER 8 ===
attn  | mean ΔF54316 = -0.0738 | mean |Δ| = 0.0874
mlp   | mean ΔF54316 = -0.2811 | mean |Δ| = 0.2811

=== LAYER 15 ===
attn  | mean ΔF54316 = -0.1848 | mean |Δ| = 0.1848
mlp   | mean ΔF54316 = -0.2013 | mean |Δ| = 0.2013

=== LAYER 9 ===
attn  | mean ΔF54316 = -0.1333 | mean |Δ| = 0.1333
mlp   | mean ΔF54316 = -0.1105 | mean |Δ| = 0.1122

=== LAYER 11 ===
attn  | mean ΔF54316 = -0.2047 | mean |Δ| = 0.2047
mlp   | mean ΔF54316 = +0.0512 | mean |Δ| = 0.0859

Component result shape: (112, 7)


### Component-Level Result

Layer-level ablation identified several candidate upstream regions, including
Layers 17 and 18.

Component-level ablation provides a more localized picture: **Layer-18
attention produces the largest measured component-level reduction in F54316
(mean |ΔF54316| = 0.4548)** among the tested components.

Layer-17 MLP is the next strongest component (mean |ΔF54316| = 0.3662).

We therefore proceed to **Layer-18 attention head decomposition** to determine
whether the component-level effect can be localized to specific attention
heads.

In [ ]:
# ============================================================
# COMPONENT RANKING
# ============================================================

component_summary = (
    component_df
    .groupby(
        ["layer", "component"]
    )
    .agg(
        mean_delta=("delta", "mean"),
        mean_abs_delta=("abs_delta", "mean"),
        std_delta=("delta", "std"),
        n=("delta", "count"),
    )
    .sort_values(
        "mean_abs_delta",
        ascending=False
    )
    .reset_index()
)

print("=== UPSTREAM COMPONENT RANKING ===")
display(component_summary)

=== UPSTREAM COMPONENT RANKING ===


,layer,component,mean_delta,mean_abs_delta,std_delta,n
0,18,attn,-0.454813,0.454813,0.043197,8
1,17,mlp,-0.366199,0.366199,0.053871,8
2,8,mlp,-0.281127,0.281127,0.053496,8
3,6,mlp,-0.237387,0.237387,0.065025,8
4,17,attn,-0.218498,0.218498,0.056566,8
5,11,attn,-0.204666,0.204666,0.089954,8
6,15,mlp,-0.201317,0.201317,0.067402,8
7,6,attn,-0.190831,0.190831,0.070995,8
8,15,attn,-0.184825,0.184825,0.036780,8
9,9,attn,-0.133326,0.133345,0.078606,8


## 8.3 Identifying Individual Attention Heads

Layer 18 is decomposed into individual attention heads.

We test the heads independently to identify which heads contribute most
strongly to F54316 activation.

In [ ]:
# ============================================================
# CIRCUIT TRACING — LAYER 18 ATTENTION HEAD SCREEN
# ============================================================

TRACE_LAYER = 18
FEATURE_ID = 54316

# Llama 3.1 8B has:
# hidden_size = 4096
# num_attention_heads = 32
# num_key_value_heads = 8
#
# Because this is GQA, multiple query heads share KV heads.
# We will first identify the query-head contributions.

NUM_Q_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_Q_HEADS

print("Layer:", TRACE_LAYER)
print("Query heads:", NUM_Q_HEADS)
print("Head dimension:", HEAD_DIM)

Layer: 18
Query heads: 32
Head dimension: 128


In [ ]:
# ============================================================
# CIRCUIT TRACING — EXACT LAYER 18 HEAD ABLATION
# ============================================================

TRACE_LAYER = 18
FEATURE_ID = 54316

NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS

assert NUM_HEADS == 32
assert HEAD_DIM == 128


def get_feature_with_head_ablation(
    word,
    layer_idx=TRACE_LAYER,
    head_idx=None,
):
    """
    Measure F54316 at Layer 19.

    If head_idx is given, zero that attention head's output
    at the final token immediately BEFORE o_proj.
    """

    captured = {}

    # --------------------------------------------------------
    # Capture Layer-19 residual-post
    # --------------------------------------------------------

    def layer19_hook(module, inputs, output):

        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        captured["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    h19 = model.model.layers[19].register_forward_hook(
        layer19_hook
    )

    # --------------------------------------------------------
    # Hook BEFORE Layer-18 o_proj
    # --------------------------------------------------------

    o_proj = model.model.layers[
        layer_idx
    ].self_attn.o_proj

    def o_proj_pre_hook(module, args):

        hidden = args[0]

        # hidden: [batch, seq, 4096]
        modified = hidden.clone()

        if head_idx is not None:

            if not (
                0 <= head_idx < NUM_HEADS
            ):
                raise ValueError(
                    f"head_idx must be "
                    f"0..{NUM_HEADS - 1}"
                )

            start = head_idx * HEAD_DIM
            end = start + HEAD_DIM

            # Ablate only the selected head
            # at the final token.
            modified[
                :, -1, start:end
            ] = 0.0

        return (modified,) + tuple(args[1:])

    ho = o_proj.register_forward_pre_hook(
        o_proj_pre_hook
    )

    try:

        messages = [
            {
                "role": "user",
                "content": make_prompt(word),
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():

            _ = model(
                **inputs,
                use_cache=False,
            )

    finally:

        ho.remove()
        h19.remove()

    # --------------------------------------------------------
    # SAE encode Layer-19 residual
    # --------------------------------------------------------

    x = captured["x"]

    with torch.no_grad():

        z = sae.encode(x)

    return z[0, FEATURE_ID].item()

In [ ]:
# ============================================================
# SANITY CHECK
# ============================================================

test_word = TRACE_WORDS[0]

baseline = get_feature_with_head_ablation(
    test_word,
    head_idx=None,
)

head0 = get_feature_with_head_ablation(
    test_word,
    head_idx=0,
)

print("Word:", test_word)
print(f"Baseline F54316 : {baseline:.4f}")
print(f"Head 0 ablated  : {head0:.4f}")
print(f"Delta            : {head0 - baseline:+.4f}")

Word: panther
Baseline F54316 : 0.9067
Head 0 ablated  : 0.7925
Delta            : -0.1142


In [ ]:
# ============================================================
# ALL 32 HEADS — LAYER 18
# ============================================================

head_rows = []

for head_idx in range(NUM_HEADS):

    effects = []

    for word in TRACE_WORDS:

        baseline = baseline_trace[word]

        ablated = get_feature_with_head_ablation(
            word,
            layer_idx=TRACE_LAYER,
            head_idx=head_idx,
        )

        delta = ablated - baseline

        effects.append(delta)

        head_rows.append({
            "layer": TRACE_LAYER,
            "head": head_idx,
            "word": word,
            "baseline": baseline,
            "ablated": ablated,
            "delta": delta,
            "abs_delta": abs(delta),
        })

    print(
        f"Head {head_idx:2d} | "
        f"mean ΔF54316 = {np.mean(effects):+.4f} | "
        f"mean |Δ| = {np.mean(np.abs(effects)):.4f}"
    )


head_ablation_df = pd.DataFrame(
    head_rows
)

print(
    "\nHead-ablation shape:",
    head_ablation_df.shape
)

Head  0 | mean ΔF54316 = -0.1571 | mean |Δ| = 0.1571
Head  1 | mean ΔF54316 = -0.0069 | mean |Δ| = 0.0077
Head  2 | mean ΔF54316 = +0.0033 | mean |Δ| = 0.0093
Head  3 | mean ΔF54316 = -0.0080 | mean |Δ| = 0.0109
Head  4 | mean ΔF54316 = -0.0036 | mean |Δ| = 0.0069
Head  5 | mean ΔF54316 = -0.0083 | mean |Δ| = 0.0121
Head  6 | mean ΔF54316 = -0.0045 | mean |Δ| = 0.0068
Head  7 | mean ΔF54316 = -0.0028 | mean |Δ| = 0.0107
Head  8 | mean ΔF54316 = -0.1704 | mean |Δ| = 0.1704
Head  9 | mean ΔF54316 = -0.0101 | mean |Δ| = 0.0116
Head 10 | mean ΔF54316 = +0.0110 | mean |Δ| = 0.0116
Head 11 | mean ΔF54316 = -0.0135 | mean |Δ| = 0.0148
Head 12 | mean ΔF54316 = -0.0095 | mean |Δ| = 0.0130
Head 13 | mean ΔF54316 = -0.0156 | mean |Δ| = 0.0156
Head 14 | mean ΔF54316 = -0.0048 | mean |Δ| = 0.0058
Head 15 | mean ΔF54316 = -0.0147 | mean |Δ| = 0.0149
Head 16 | mean ΔF54316 = -0.0061 | mean |Δ| = 0.0092
Head 17 | mean ΔF54316 = -0.0026 | mean |Δ| = 0.0061
Head 18 | mean ΔF54316 = -0.0071 | mean |Δ| = 

In [ ]:
# ============================================================
# ABLATE ONE ATTENTION HEAD AT FINAL POSITION
# ============================================================

def get_feature_with_head_ablation(
    word,
    layer_idx,
    head_idx,
):
    """
    Ablate one query attention head's contribution to the
    attention output at the final token position.
    """

    captured = {}

    # --------------------------------------------------------
    # Capture Layer 19 residual
    # --------------------------------------------------------

    def layer19_hook(module, inputs, output):
        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        captured["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    h19 = model.model.layers[19].register_forward_hook(
        layer19_hook
    )

    # --------------------------------------------------------
    # Attention hook
    # --------------------------------------------------------

    attn_module = model.model.layers[
        layer_idx
    ].self_attn

    def head_hook(module, inputs, output):

        # LlamaAttention output[0] is [batch, seq, hidden]
        attn_output = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        modified = attn_output.clone()

        # IMPORTANT:
        # o_proj mixes all heads, so the direct output-space
        # contribution of one head is not explicitly available
        # here. We therefore use a zeroing approximation at the
        # attention-output level by reconstructing head output
        # when possible from module internals.

        # For this first screen, do not silently pretend this is
        # exact head ablation.
        raise RuntimeError(
            "Head decomposition requires hooking before o_proj. "
            "Use the next cell below."
        )

    # We don't actually register this hook.
    h19.remove()

In [ ]:
# ============================================================
# HEAD RANKING
# ============================================================

head_summary = (
    head_ablation_df
    .groupby(["layer", "head"])
    .agg(
        mean_delta=("delta", "mean"),
        mean_abs_delta=("abs_delta", "mean"),
        std_delta=("delta", "std"),
        n=("delta", "count"),
    )
    .sort_values(
        "mean_abs_delta",
        ascending=False
    )
    .reset_index()
)

print("=== LAYER 18 HEAD RANKING ===")
display(head_summary)

=== LAYER 18 HEAD RANKING ===


,layer,head,mean_delta,mean_abs_delta,std_delta,n
0,18,8,-0.170433,0.170433,0.016358,8
1,18,0,-0.157149,0.157149,0.035432,8
2,18,26,0.036788,0.036788,0.012985,8
3,18,24,-0.023698,0.025875,0.020238,8
4,18,25,-0.023844,0.023844,0.016587,8
5,18,22,-0.022424,0.022808,0.014569,8
6,18,13,-0.015631,0.015631,0.008089,8
7,18,20,0.007216,0.014913,0.017080,8
8,18,15,-0.014660,0.014876,0.009746,8
9,18,11,-0.013520,0.014753,0.013089,8


## 8.4 Testing Candidate Heads on Behavior

The strongest F54316-contributing heads are now tested directly on the
animal-vs-vehicle behavioral margin.

This determines whether a head that affects F54316 also affects the behavior
of interest.

In [ ]:
# ============================================================
# CIRCUIT: L18 HEAD -> BEHAVIOR TEST
# ============================================================

CIRCUIT_HEADS = [8, 0]

def behavioral_margin_with_head_ablation(
    word,
    head_idx,
    layer_idx=18,
):
    """
    Animal-vs-vehicle behavioral margin after ablating
    one Layer-18 attention head at the final token.
    """

    prompt = make_prompt(word)

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    prefix_len = inputs["input_ids"].shape[1]

    # --------------------------------------------------------
    # Continuation tokens
    # --------------------------------------------------------

    cont_a = tokenizer(
        BEHAVIOR["cont_a"],
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    cont_b = tokenizer(
        BEHAVIOR["cont_b"],
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    # --------------------------------------------------------
    # Build full sequence for A/B
    # --------------------------------------------------------

    def run_margin(continuation_ids):

        full_ids = torch.cat(
            [
                inputs["input_ids"],
                continuation_ids,
            ],
            dim=1,
        )

        def o_proj_pre_hook(module, args):

            hidden = args[0].clone()

            start = head_idx * HEAD_DIM
            end = start + HEAD_DIM

            hidden[:, prefix_len - 1, start:end] = 0.0

            return (hidden,) + tuple(args[1:])

        target = model.model.layers[
            layer_idx
        ].self_attn.o_proj

        handle = target.register_forward_pre_hook(
            o_proj_pre_hook
        )

        try:
            with torch.no_grad():
                logits = model(
                    input_ids=full_ids,
                    use_cache=False,
                ).logits
        finally:
            handle.remove()

        start = prefix_len - 1
        end = full_ids.shape[1] - 1

        log_probs = torch.log_softmax(
            logits[0, start:end],
            dim=-1,
        )

        token_logps = log_probs.gather(
            1,
            continuation_ids[0].unsqueeze(-1),
        ).squeeze(-1)

        return token_logps.sum().item()

    lp_a = run_margin(cont_a)
    lp_b = run_margin(cont_b)

    return lp_a - lp_b

In [ ]:
# ============================================================
# RUN ON FINAL CAUSAL TEST SET
# ============================================================

circuit_rows = []

for word, label in final_causal_words:

    baseline = final_baselines[word]

    for head_idx in CIRCUIT_HEADS:

        ablated_margin = (
            behavioral_margin_with_head_ablation(
                word,
                head_idx,
            )
        )

        delta_m = ablated_margin - baseline

        circuit_rows.append({
            "word": word,
            "label": label,
            "head": head_idx,
            "baseline_margin": baseline,
            "ablated_margin": ablated_margin,
            "delta_margin": delta_m,
        })

circuit_behavior_df = pd.DataFrame(
    circuit_rows
)

display(circuit_behavior_df)

,word,label,head,baseline_margin,ablated_margin,delta_margin
0,leopard,animal,8,12.375,11.750,-0.625
1,leopard,animal,0,12.375,12.125,-0.250
2,cheetah,animal,8,11.250,11.000,-0.250
3,cheetah,animal,0,11.250,11.750,0.500
4,hyena,animal,8,12.250,11.750,-0.500
5,hyena,animal,0,12.250,12.250,0.000
6,rhinoceros,animal,8,15.750,15.875,0.125
7,rhinoceros,animal,0,15.750,16.875,1.125
8,hippopotamus,animal,8,17.875,17.750,-0.125
9,hippopotamus,animal,0,17.875,18.500,0.625


In [ ]:
# ============================================================
# HEAD -> BEHAVIOR SUMMARY
# ============================================================

circuit_behavior_summary = (
    circuit_behavior_df
    .groupby(["head", "label"])
    .agg(
        mean_delta=("delta_margin", "mean"),
        std_delta=("delta_margin", "std"),
        positive_fraction=(
            "delta_margin",
            lambda x: (x > 0).mean()
        ),
        n=("delta_margin", "count"),
    )
    .reset_index()
)

print("=== L18 HEAD -> BEHAVIOR ===")
display(circuit_behavior_summary)

=== L18 HEAD -> BEHAVIOR ===


,head,label,mean_delta,std_delta,positive_fraction,n
0,0,animal,0.1750,0.534244,0.5,10
1,0,vehicle,0.2375,0.587633,0.6,10
2,8,animal,-0.2000,0.486627,0.3,10
3,8,vehicle,0.2250,0.617454,0.7,10


## 8.5 Feature-Mediated Rescue

Head ablation establishes that a candidate head affects F54316, but does not
by itself establish that the behavioral effect is mediated through F54316.

We therefore ablate the candidate head and restore the F54316 feature direction.

The intended causal pathway is:

```text
Upstream head
      ↓
  F54316
      ↓
  Behavior

In [ ]:
# ============================================================
# FIXED: H8 -> F54316 MEDIATION VALIDATION
# ============================================================

FEATURE_ID = 54316
HEAD_ID = 8
TRACE_LAYER = 18

decoder = (
    sae.W_dec[FEATURE_ID]
    .detach()
    .clone()
    .float()
)

decoder = decoder / decoder.norm().clamp_min(1e-8)

o_proj = model.model.layers[
    TRACE_LAYER
].self_attn.o_proj


def measure_h8_rescue_state(word, head_idx=HEAD_ID):

    # --------------------------------------------------------
    # 1. Baseline feature
    # --------------------------------------------------------

    baseline_x = get_layer19_activation(word)

    with torch.no_grad():
        baseline_z = sae.encode(baseline_x)

    baseline_feature = baseline_z[
        0, FEATURE_ID
    ].item()

    baseline_margin = final_baselines[word]

    # --------------------------------------------------------
    # 2. Measure F54316 after H8 ablation
    # --------------------------------------------------------

    captured = {}

    def capture_layer19(module, inputs, output):

        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        captured["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    def h8_hook(module, args):

        hidden = args[0].clone()

        start = head_idx * HEAD_DIM
        end = start + HEAD_DIM

        hidden[:, -1, start:end] = 0.0

        return (hidden,) + tuple(args[1:])

    h19 = model.model.layers[
        19
    ].register_forward_hook(
        capture_layer19
    )

    hh = o_proj.register_forward_pre_hook(
        h8_hook
    )

    try:

        messages = [
            {
                "role": "user",
                "content": make_prompt(word),
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            _ = model(
                **inputs,
                use_cache=False,
            )

    finally:

        hh.remove()
        h19.remove()

    with torch.no_grad():
        ablated_z = sae.encode(
            captured["x"]
        )

    ablated_feature = ablated_z[
        0, FEATURE_ID
    ].item()

    # Amount needed to restore original SAE feature
    rescue_alpha = max(
        0.0,
        baseline_feature - ablated_feature
    )

    # --------------------------------------------------------
    # 3. H8 ablation + F54316 rescue
    # --------------------------------------------------------

    captured_rescue = {}

    def h8_rescue_hook(module, args):

        hidden = args[0].clone()

        start = head_idx * HEAD_DIM
        end = start + HEAD_DIM

        hidden[:, -1, start:end] = 0.0

        return (hidden,) + tuple(args[1:])

    def layer19_rescue_hook(module, inputs, output):

        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        modified = hidden.clone()

        if rescue_alpha > 0:

            d = decoder.to(
                device=modified.device,
                dtype=modified.dtype,
            )

            modified[:, -1, :] += (
                rescue_alpha * d
            )

        if isinstance(output, tuple):
            return (
                (modified,)
                + output[1:]
            )

        return modified

    def capture_rescued_layer19(
        module,
        inputs,
        output,
    ):

        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        captured_rescue["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    hh2 = o_proj.register_forward_pre_hook(
        h8_rescue_hook
    )

    hr = model.model.layers[
        19
    ].register_forward_hook(
        layer19_rescue_hook
    )

    hc = model.model.layers[
        19
    ].register_forward_hook(
        capture_rescued_layer19
    )

    try:

        with torch.no_grad():
            _ = model(
                **inputs,
                use_cache=False,
            )

    finally:

        hh2.remove()
        hr.remove()
        hc.remove()

    # --------------------------------------------------------
    # 4. Measure actual rescued SAE activation
    # --------------------------------------------------------

    with torch.no_grad():

        rescued_z = sae.encode(
            captured_rescue["x"]
        )

    rescued_feature = rescued_z[
        0, FEATURE_ID
    ].item()

    # --------------------------------------------------------
    # 5. Behavioral margin after rescue
    # --------------------------------------------------------

    def run_behavioral_margin(
        rescue_enabled=True,
    ):

        prompt = make_prompt(word)

        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]

        prefix_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
        )

        if not torch.is_tensor(prefix_ids):
            prefix_ids = prefix_ids.input_ids

        prefix_ids = prefix_ids.to(model.device)

        cont_a = tokenizer(
            BEHAVIOR["cont_a"],
            add_special_tokens=False,
            return_tensors="pt",
        ).input_ids.to(model.device)

        cont_b = tokenizer(
            BEHAVIOR["cont_b"],
            add_special_tokens=False,
            return_tensors="pt",
        ).input_ids.to(model.device)

        prefix_len = prefix_ids.shape[1]

        def score(cont_ids):

            full_ids = torch.cat(
                [prefix_ids, cont_ids],
                dim=1,
            )

            def head_hook(module, args):

                hidden = args[0].clone()

                start = head_idx * HEAD_DIM
                end = start + HEAD_DIM

                hidden[:, -1, start:end] = 0.0

                return (
                    hidden,
                ) + tuple(args[1:])

            def rescue_hook(module, inputs, output):

                hidden = (
                    output[0]
                    if isinstance(output, tuple)
                    else output
                )

                modified = hidden.clone()

                if (
                    rescue_enabled
                    and rescue_alpha > 0
                ):

                    d = decoder.to(
                        device=modified.device,
                        dtype=modified.dtype,
                    )

                    modified[:, -1, :] += (
                        rescue_alpha * d
                    )

                if isinstance(output, tuple):

                    return (
                        (modified,)
                        + output[1:]
                    )

                return modified

            hh = o_proj.register_forward_pre_hook(
                head_hook
            )

            hr = model.model.layers[
                19
            ].register_forward_hook(
                rescue_hook
            )

            try:

                with torch.no_grad():

                    logits = model(
                        input_ids=full_ids,
                        use_cache=False,
                    ).logits

            finally:

                hh.remove()
                hr.remove()

            start = prefix_len - 1
            end = full_ids.shape[1] - 1

            log_probs = torch.log_softmax(
                logits[0, start:end],
                dim=-1,
            )

            token_logps = log_probs.gather(
                1,
                cont_ids[0].unsqueeze(-1),
            ).squeeze(-1)

            return token_logps.sum().item()

        lp_a = score(cont_a)
        lp_b = score(cont_b)

        return lp_a - lp_b

    rescued_margin = run_behavioral_margin(
        rescue_enabled=True
    )

    return {
        "baseline_feature": baseline_feature,
        "ablated_feature": ablated_feature,
        "rescued_feature": rescued_feature,
        "feature_drop": (
            ablated_feature
            - baseline_feature
        ),
        "feature_residual_error": (
            rescued_feature
            - baseline_feature
        ),
        "rescue_alpha": rescue_alpha,
        "baseline_margin": baseline_margin,
        "rescued_margin": rescued_margin,
        "rescued_delta": (
            rescued_margin
            - baseline_margin
        ),
    }

In [ ]:
# ============================================================
# RUN H8 MEDIATION VALIDATION
# ============================================================

validation_rows = []

for word, label in final_causal_words:

    result = measure_h8_rescue_state(
        word,
        head_idx=8,
    )

    result["word"] = word
    result["label"] = label

    validation_rows.append(result)


h8_mediation_validation = pd.DataFrame(
    validation_rows
)

display(h8_mediation_validation)

,baseline_feature,ablated_feature,rescued_feature,feature_drop,feature_residual_error,rescue_alpha,baseline_margin,rescued_margin,rescued_delta,word,label
0,1.144971,0.979543,1.133443,-0.165428,-0.011528,0.165428,12.375,12.375,0.000,leopard,animal
1,1.264655,1.108258,1.252517,-0.156398,-0.012138,0.156398,11.250,11.250,0.000,cheetah,animal
2,1.425151,1.198425,1.408210,-0.226725,-0.016940,0.226725,12.250,12.250,0.000,hyena,animal
3,1.240067,1.017893,1.223325,-0.222174,-0.016741,0.222174,15.750,15.750,0.000,rhinoceros,animal
4,1.506752,1.310756,1.492076,-0.195996,-0.014676,0.195996,17.875,17.875,0.000,hippopotamus,animal
5,0.694284,0.532599,0.682255,-0.161685,-0.012029,0.161685,13.750,13.750,0.000,buffalo,animal
6,1.164939,0.988156,1.151121,-0.176782,-0.013818,0.176782,12.625,12.625,0.000,antelope,animal
7,0.576182,0.468905,0.567842,-0.107277,-0.008340,0.107277,10.750,10.750,0.000,squirrel,animal
8,1.281159,1.101741,1.267502,-0.179417,-0.013656,0.179417,6.375,6.375,0.000,flamingo,animal
9,1.454704,1.278843,1.441277,-0.175861,-0.013427,0.175861,15.500,15.500,0.000,meerkat,animal


In [ ]:
# ============================================================
# SUMMARY
# ============================================================

animal = h8_mediation_validation[
    h8_mediation_validation["label"]
    == BEHAVIOR["cat_a"]
]

vehicle = h8_mediation_validation[
    h8_mediation_validation["label"]
    == BEHAVIOR["cat_b"]
]

print("=== H8 MEDIATION VALIDATION ===")

print("\nANIMALS")
print(
    "Mean feature drop:",
    animal["feature_drop"].mean()
)
print(
    "Mean rescue error:",
    animal["feature_residual_error"].mean()
)
print(
    "Mean rescue delta:",
    animal["rescued_delta"].mean()
)

print("\nVEHICLES")
print(
    "Mean feature drop:",
    vehicle["feature_drop"].mean()
)
print(
    "Mean rescue error:",
    vehicle["feature_residual_error"].mean()
)
print(
    "Mean rescue delta:",
    vehicle["rescued_delta"].mean()
)

=== H8 MEDIATION VALIDATION ===

ANIMALS
Mean feature drop: -0.17677417993545533
Mean rescue error: -0.013329458236694337
Mean rescue delta: 0.0

VEHICLES
Mean feature drop: 0.0
Mean rescue error: 0.0
Mean rescue delta: 0.0125


In [ ]:
# ============================================================
# GENERIC HEAD -> F54316 -> BEHAVIOR MEDIATION
# ============================================================

FEATURE_ID = 54316
TRACE_LAYER = 18

decoder = (
    sae.W_dec[FEATURE_ID]
    .detach()
    .clone()
    .float()
)

decoder = decoder / decoder.norm().clamp_min(1e-8)

o_proj = model.model.layers[
    TRACE_LAYER
].self_attn.o_proj


def measure_head_rescue_state(word, head_idx):

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline_x = get_layer19_activation(word)

    with torch.no_grad():
        baseline_z = sae.encode(baseline_x)

    baseline_feature = baseline_z[0, FEATURE_ID].item()
    baseline_margin = final_baselines[word]

    # --------------------------------------------------------
    # H(head) ablation -> feature
    # --------------------------------------------------------

    captured = {}

    def capture_layer19(module, inputs, output):
        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )
        captured["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    def head_hook(module, args):
        hidden = args[0].clone()

        start = head_idx * HEAD_DIM
        end = start + HEAD_DIM

        hidden[:, -1, start:end] = 0.0

        return (hidden,) + tuple(args[1:])

    h19 = model.model.layers[19].register_forward_hook(
        capture_layer19
    )
    hh = o_proj.register_forward_pre_hook(
        head_hook
    )

    try:
        messages = [
            {
                "role": "user",
                "content": make_prompt(word),
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            _ = model(
                **inputs,
                use_cache=False,
            )
    finally:
        hh.remove()
        h19.remove()

    with torch.no_grad():
        ablated_z = sae.encode(captured["x"])

    ablated_feature = ablated_z[0, FEATURE_ID].item()

    rescue_alpha = max(
        0.0,
        baseline_feature - ablated_feature
    )

    # --------------------------------------------------------
    # H(head) ablation + F54316 rescue
    # --------------------------------------------------------

    captured_rescue = {}

    def rescue_hook(module, inputs, output):
        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        modified = hidden.clone()

        if rescue_alpha > 0:
            d = decoder.to(
                device=modified.device,
                dtype=modified.dtype,
            )

            modified[:, -1, :] += (
                rescue_alpha * d
            )

        if isinstance(output, tuple):
            return (
                (modified,)
                + output[1:]
            )

        return modified

    def capture_rescued(module, inputs, output):
        hidden = (
            output[0]
            if isinstance(output, tuple)
            else output
        )

        captured_rescue["x"] = (
            hidden[:, -1, :]
            .detach()
            .float()
            .cpu()
        )

    hh2 = o_proj.register_forward_pre_hook(head_hook)

    hr = model.model.layers[19].register_forward_hook(
        rescue_hook
    )

    hc = model.model.layers[19].register_forward_hook(
        capture_rescued
    )

    try:
        with torch.no_grad():
            _ = model(
                **inputs,
                use_cache=False,
            )
    finally:
        hh2.remove()
        hr.remove()
        hc.remove()

    with torch.no_grad():
        rescued_z = sae.encode(captured_rescue["x"])

    rescued_feature = rescued_z[0, FEATURE_ID].item()

    # --------------------------------------------------------
    # Behavioral margin after ablation + rescue
    # --------------------------------------------------------

    prompt = make_prompt(word)

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    prefix_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )

    if not torch.is_tensor(prefix_ids):
        prefix_ids = prefix_ids.input_ids

    prefix_ids = prefix_ids.to(model.device)

    cont_a = tokenizer(
        BEHAVIOR["cont_a"],
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    cont_b = tokenizer(
        BEHAVIOR["cont_b"],
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    prefix_len = prefix_ids.shape[1]

    def score(cont_ids):

        full_ids = torch.cat(
            [prefix_ids, cont_ids],
            dim=1
        )

        hh_local = o_proj.register_forward_pre_hook(
            head_hook
        )

        hr_local = model.model.layers[19].register_forward_hook(
            rescue_hook
        )

        try:
            with torch.no_grad():
                logits = model(
                    input_ids=full_ids,
                    use_cache=False,
                ).logits
        finally:
            hh_local.remove()
            hr_local.remove()

        start = prefix_len - 1
        end = full_ids.shape[1] - 1

        log_probs = torch.log_softmax(
            logits[0, start:end],
            dim=-1,
        )

        return log_probs.gather(
            1,
            cont_ids[0].unsqueeze(-1),
        ).squeeze(-1).sum().item()

    rescued_margin = (
        score(cont_a)
        - score(cont_b)
    )

    return {
        "baseline_feature": baseline_feature,
        "ablated_feature": ablated_feature,
        "rescued_feature": rescued_feature,
        "feature_drop": (
            ablated_feature - baseline_feature
        ),
        "feature_residual_error": (
            rescued_feature - baseline_feature
        ),
        "rescue_alpha": rescue_alpha,
        "baseline_margin": baseline_margin,
        "rescued_margin": rescued_margin,
        "rescued_delta": (
            rescued_margin - baseline_margin
        ),
        "head": head_idx,
        "word": word,
    }

In [ ]:
# ============================================================
# H0 MEDIATION VALIDATION
# ============================================================

HEAD_TO_TEST = 0

rows = []

for word, label in final_causal_words:

    result = measure_head_rescue_state(
        word,
        head_idx=HEAD_TO_TEST,
    )

    result["label"] = label
    rows.append(result)

h0_mediation_validation = pd.DataFrame(rows)

display(h0_mediation_validation)

,baseline_feature,ablated_feature,rescued_feature,feature_drop,feature_residual_error,rescue_alpha,baseline_margin,rescued_margin,rescued_delta,head,word,label
0,1.144971,0.989325,1.133375,-0.155646,-0.011595,0.155646,12.375,12.375,0.000,0,leopard,animal
1,1.264655,1.073906,1.249381,-0.190750,-0.015275,0.190750,11.250,11.250,0.000,0,cheetah,animal
2,1.425151,1.202336,1.408465,-0.222814,-0.016686,0.222814,12.250,12.250,0.000,0,hyena,animal
3,1.240067,1.072555,1.228364,-0.167511,-0.011703,0.167511,15.750,15.750,0.000,0,rhinoceros,animal
4,1.506752,1.263819,1.488526,-0.242933,-0.018226,0.242933,17.875,17.875,0.000,0,hippopotamus,animal
5,0.694284,0.615970,0.689511,-0.078314,-0.004773,0.078314,13.750,13.750,0.000,0,buffalo,animal
6,1.164939,1.028574,1.154142,-0.136365,-0.010797,0.136365,12.625,12.625,0.000,0,antelope,animal
7,0.576182,0.517484,0.572189,-0.058698,-0.003993,0.058698,10.750,10.750,0.000,0,squirrel,animal
8,1.281159,1.157772,1.270643,-0.123386,-0.010516,0.123386,6.375,6.375,0.000,0,flamingo,animal
9,1.454704,1.261749,1.439219,-0.192955,-0.015485,0.192955,15.500,15.500,0.000,0,meerkat,animal


In [ ]:
# ============================================================
# H0 SUMMARY
# ============================================================

animal = h0_mediation_validation[
    h0_mediation_validation["label"]
    == BEHAVIOR["cat_a"]
]

vehicle = h0_mediation_validation[
    h0_mediation_validation["label"]
    == BEHAVIOR["cat_b"]
]

print("=== H0 MEDIATION ===")

print("\nANIMALS")
print(
    "Mean feature drop:",
    animal["feature_drop"].mean()
)
print(
    "Mean rescue error:",
    animal["feature_residual_error"].mean()
)
print(
    "Mean rescued ΔM:",
    animal["rescued_delta"].mean()
)

print("\nVEHICLES")
print(
    "Mean feature drop:",
    vehicle["feature_drop"].mean()
)
print(
    "Mean rescue error:",
    vehicle["feature_residual_error"].mean()
)
print(
    "Mean rescued ΔM:",
    vehicle["rescued_delta"].mean()
)

=== H0 MEDIATION ===

ANIMALS
Mean feature drop: -0.15693721771240235
Mean rescue error: -0.011904764175415038
Mean rescued ΔM: 0.0

VEHICLES
Mean feature drop: 0.0
Mean rescue error: 0.0
Mean rescued ΔM: 0.0375


## 8.6 Downstream Mediation

We test whether later transformer layers mediate the behavioral effect of
F54316 using a 2×2 intervention design.

The analysis compares the effect of the F54316 intervention under normal and
downstream-blocked conditions.

In [ ]:
# ============================================================
# PROPER 2x2 DOWNSTREAM MEDIATION TEST
# ============================================================

def behavioral_margin_condition(
    word,
    feature_id=54316,
    fraction=0.20,
    sign=0.0,
    downstream_layer=None,
):
    """
    sign = 0.0 -> no F54316 intervention
    sign = +1.0 -> increase F54316
    sign = -1.0 -> suppress F54316

    downstream_layer = None -> normal
    downstream_layer = integer -> ablate that layer
    """

    prompt = make_prompt(word)

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    prefix_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )

    if not torch.is_tensor(prefix_ids):
        prefix_ids = prefix_ids.input_ids

    prefix_ids = prefix_ids.to(model.device)

    cont_a = tokenizer(
        BEHAVIOR["cont_a"],
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    cont_b = tokenizer(
        BEHAVIOR["cont_b"],
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(model.device)

    prefix_len = prefix_ids.shape[1]
    target_pos = prefix_len - 1

    strength = fraction * ref_norm

    decoder = (
        sae.W_dec[feature_id]
        .detach()
        .clone()
        .float()
    )

    decoder = decoder / decoder.norm().clamp_min(1e-8)

    def score(cont_ids):

        full_ids = torch.cat(
            [prefix_ids, cont_ids],
            dim=1,
        )

        handles = []

        # ----------------------------------------------------
        # F54316 intervention at Layer 19
        # ----------------------------------------------------

        if sign != 0.0:

            def feature_hook(module, inputs, output):

                hidden = (
                    output[0]
                    if isinstance(output, tuple)
                    else output
                )

                modified = hidden.clone()

                d = decoder.to(
                    device=modified.device,
                    dtype=modified.dtype,
                )

                modified[:, target_pos, :] += (
                    sign * strength * d
                )

                if isinstance(output, tuple):
                    return (
                        (modified,)
                        + output[1:]
                    )

                return modified

            handles.append(
                model.model.layers[
                    19
                ].register_forward_hook(
                    feature_hook
                )
            )

        # ----------------------------------------------------
        # Downstream layer ablation
        # ----------------------------------------------------

        if downstream_layer is not None:

            def downstream_hook(
                module,
                inputs,
                output,
            ):

                hidden_in = inputs[0]

                if isinstance(output, tuple):

                    hidden_out = output[0].clone()

                    hidden_out[:, target_pos, :] = (
                        hidden_in[:, target_pos, :]
                    )

                    return (
                        (hidden_out,)
                        + output[1:]
                    )

                hidden_out = output.clone()

                hidden_out[:, target_pos, :] = (
                    hidden_in[:, target_pos, :]
                )

                return hidden_out

            handles.append(
                model.model.layers[
                    downstream_layer
                ].register_forward_hook(
                    downstream_hook
                )
            )

        try:

            with torch.no_grad():

                logits = model(
                    input_ids=full_ids,
                    use_cache=False,
                ).logits

        finally:

            for h in handles:
                h.remove()

        start = prefix_len - 1
        end = full_ids.shape[1] - 1

        log_probs = torch.log_softmax(
            logits[0, start:end],
            dim=-1,
        )

        token_logps = log_probs.gather(
            1,
            cont_ids[0].unsqueeze(-1),
        ).squeeze(-1)

        return token_logps.sum().item()

    lp_a = score(cont_a)
    lp_b = score(cont_b)

    return lp_a - lp_b

In [ ]:
# ============================================================
# PROPER 2x2 MEDIATION
# ============================================================

DOWNSTREAM_CANDIDATES = [26, 28, 29, 25]

rows = []

for word, label in final_causal_words:

    # --------------------------------------------
    # Normal baseline
    # --------------------------------------------

    A = behavioral_margin_condition(
        word,
        feature_id=54316,
        fraction=0.20,
        sign=0.0,
        downstream_layer=None,
    )

    # --------------------------------------------
    # Normal + F54316 increase
    # --------------------------------------------

    B = behavioral_margin_condition(
        word,
        feature_id=54316,
        fraction=0.20,
        sign=+1.0,
        downstream_layer=None,
    )

    normal_effect = B - A

    for layer_idx in DOWNSTREAM_CANDIDATES:

        # ----------------------------------------
        # Downstream ablation only
        # ----------------------------------------

        C = behavioral_margin_condition(
            word,
            feature_id=54316,
            fraction=0.20,
            sign=0.0,
            downstream_layer=layer_idx,
        )

        # ----------------------------------------
        # Downstream ablation + F54316 increase
        # ----------------------------------------

        D = behavioral_margin_condition(
            word,
            feature_id=54316,
            fraction=0.20,
            sign=+1.0,
            downstream_layer=layer_idx,
        )

        blocked_effect = D - C

        interaction = (
            normal_effect
            - blocked_effect
        )

        rows.append({
            "word": word,
            "label": label,
            "layer": layer_idx,
            "A_baseline": A,
            "B_feature": B,
            "C_ablation": C,
            "D_feature_plus_ablation": D,
            "normal_effect": normal_effect,
            "blocked_effect": blocked_effect,
            "interaction": interaction,
        })

proper_downstream_mediation = pd.DataFrame(rows)

print(
    "Shape:",
    proper_downstream_mediation.shape
)

display(
    proper_downstream_mediation.head(12)
)

Shape: (80, 10)


,word,label,layer,A_baseline,B_feature,C_ablation,D_feature_plus_ablation,normal_effect,blocked_effect,interaction
0,leopard,animal,26,12.375,12.50,10.125,11.750,0.125,1.625,-1.500
1,leopard,animal,28,12.375,12.50,12.375,12.500,0.125,0.125,0.000
2,leopard,animal,29,12.375,12.50,13.875,14.125,0.125,0.250,-0.125
3,leopard,animal,25,12.375,12.50,13.000,14.000,0.125,1.000,-0.875
4,cheetah,animal,26,11.250,12.25,10.875,13.375,1.000,2.500,-1.500
5,cheetah,animal,28,11.250,12.25,12.750,14.125,1.000,1.375,-0.375
6,cheetah,animal,29,11.250,12.25,13.000,13.750,1.000,0.750,0.250
7,cheetah,animal,25,11.250,12.25,11.875,13.000,1.000,1.125,-0.125
8,hyena,animal,26,12.250,12.50,11.625,11.500,0.250,-0.125,0.375
9,hyena,animal,28,12.250,12.50,13.250,14.125,0.250,0.875,-0.625


In [ ]:
# ============================================================
# PROPER MEDIATION SUMMARY
# ============================================================

proper_summary = (
    proper_downstream_mediation
    .groupby(["layer", "label"])
    .agg(
        normal_effect=("normal_effect", "mean"),
        blocked_effect=("blocked_effect", "mean"),
        interaction=("interaction", "mean"),
        std_interaction=("interaction", "std"),
        n=("interaction", "count"),
    )
    .reset_index()
)

print("=== PROPER DOWNSTREAM MEDIATION ===")
display(proper_summary)

=== PROPER DOWNSTREAM MEDIATION ===


,layer,label,normal_effect,blocked_effect,interaction,std_interaction,n
0,25,animal,0.8500,1.6625,-0.8125,0.917140,10
1,25,vehicle,0.6875,0.9250,-0.2375,0.670432,10
2,26,animal,0.8500,1.0750,-0.2250,0.801041,10
3,26,vehicle,0.6875,1.1750,-0.4875,0.673017,10
4,28,animal,0.8500,1.2625,-0.4125,0.664188,10
5,28,vehicle,0.6875,0.6750,0.0125,0.698336,10
6,29,animal,0.8500,1.2625,-0.4125,0.486091,10
7,29,vehicle,0.6875,1.1875,-0.5000,0.750000,10


## 8.7 Downstream Perturbation Propagation

We next track the effect of the F54316 intervention through Layers 19–31.

For each layer, we measure the perturbation magnitude, relative perturbation,
and similarity between the baseline and intervened representations.

In [ ]:
# ============================================================
# F54316 DOWNSTREAM PROPAGATION TRACE
# ============================================================

TRACE_LAYERS = list(range(19, 32))
TRACE_WORDS = [
    "leopard",
    "cheetah",
    "hyena",
    "rhinoceros",
    "hippopotamus",
    "buffalo",
    "antelope",
    "squirrel",
    "flamingo",
    "meerkat",
]

FEATURE_ID = 54316
TRACE_FRACTION = 0.20

In [ ]:
# ============================================================
# CAPTURE BASELINE VS F54316-INTERVENED STATES
# ============================================================

def capture_layer_states(
    word,
    intervene=False,
    fraction=0.20,
):
    """
    Capture the final-token hidden state after every layer
    from 19 through 31.

    intervene=True adds +F54316 decoder direction at Layer 19.
    """

    states = {}

    decoder = (
        sae.W_dec[FEATURE_ID]
        .detach()
        .clone()
        .float()
    )

    decoder = decoder / decoder.norm().clamp_min(1e-8)

    strength = fraction * ref_norm

    # --------------------------------------------------------
    # Capture layers 19-31
    # --------------------------------------------------------

    handles = []

    for layer_idx in TRACE_LAYERS:

        def make_hook(idx):

            def hook(module, inputs, output):

                hidden = (
                    output[0]
                    if isinstance(output, tuple)
                    else output
                )

                states[idx] = (
                    hidden[:, -1, :]
                    .detach()
                    .float()
                    .cpu()
                )

            return hook

        handles.append(
            model.model.layers[layer_idx]
            .register_forward_hook(
                make_hook(layer_idx)
            )
        )

    # --------------------------------------------------------
    # Optional F54316 intervention at Layer 19
    # --------------------------------------------------------

    intervention_handle = None

    if intervene:

        def feature_hook(module, inputs, output):

            hidden = (
                output[0]
                if isinstance(output, tuple)
                else output
            )

            modified = hidden.clone()

            d = decoder.to(
                device=modified.device,
                dtype=modified.dtype,
            )

            # IMPORTANT:
            # Last prompt token = position -1 here because
            # this forward pass contains only the prompt
            # at the point where the Layer-19 hook fires.
            modified[:, -1, :] += (
                strength * d
            )

            if isinstance(output, tuple):
                return (
                    (modified,)
                    + output[1:]
                )

            return modified

        intervention_handle = (
            model.model.layers[19]
            .register_forward_hook(
                feature_hook
            )
        )

    try:

        messages = [
            {
                "role": "user",
                "content": make_prompt(word),
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():

            _ = model(
                **inputs,
                use_cache=False,
            )

    finally:

        for h in handles:
            h.remove()

        if intervention_handle is not None:
            intervention_handle.remove()

    return states

In [ ]:
# ============================================================
# RUN PROPAGATION TRACE
# ============================================================

prop_rows = []

for word in TRACE_WORDS:

    baseline_states = capture_layer_states(
        word,
        intervene=False,
    )

    intervention_states = capture_layer_states(
        word,
        intervene=True,
        fraction=TRACE_FRACTION,
    )

    for layer_idx in TRACE_LAYERS:

        x0 = baseline_states[layer_idx]
        x1 = intervention_states[layer_idx]

        delta = x1 - x0

        delta_norm = delta.norm().item()

        base_norm = x0.norm().item()

        relative_delta = (
            delta_norm
            / max(base_norm, 1e-8)
        )

        cosine = torch.nn.functional.cosine_similarity(
            x0,
            x1,
            dim=-1,
        ).item()

        prop_rows.append({
            "word": word,
            "layer": layer_idx,
            "delta_norm": delta_norm,
            "baseline_norm": base_norm,
            "relative_delta": relative_delta,
            "cosine": cosine,
        })

propagation_df = pd.DataFrame(
    prop_rows
)

print(
    "Propagation shape:",
    propagation_df.shape
)

display(propagation_df.head(20))

Propagation shape: (130, 6)


,word,layer,delta_norm,baseline_norm,relative_delta,cosine
0,leopard,19,0.000000,12.733535,0.000000,1.000001
1,leopard,20,3.794899,13.468402,0.281763,0.966164
2,leopard,21,4.328071,15.745543,0.274876,0.966117
3,leopard,22,4.409543,17.783508,0.247957,0.970853
4,leopard,23,4.597550,19.822683,0.231934,0.974115
5,leopard,24,4.895154,21.463636,0.228067,0.975228
6,leopard,25,5.264730,23.605669,0.223028,0.976326
7,leopard,26,5.540142,26.934641,0.205688,0.979840
8,leopard,27,6.013272,28.616810,0.210131,0.979050
9,leopard,28,6.466556,32.521736,0.198838,0.981041


In [ ]:
# ============================================================
# PROPAGATION SUMMARY
# ============================================================

propagation_summary = (
    propagation_df
    .groupby("layer")
    .agg(
        mean_delta_norm=("delta_norm", "mean"),
        std_delta_norm=("delta_norm", "std"),
        mean_relative_delta=("relative_delta", "mean"),
        mean_cosine=("cosine", "mean"),
    )
    .reset_index()
)

print("=== F54316 DOWNSTREAM PROPAGATION ===")
display(propagation_summary)

=== F54316 DOWNSTREAM PROPAGATION ===


,layer,mean_delta_norm,std_delta_norm,mean_relative_delta,mean_cosine
0,19,0.000000,0.000000,0.000000,1.000001
1,20,3.735575,0.127544,0.276891,0.967402
2,21,4.261859,0.128073,0.268570,0.967441
3,22,4.372530,0.101306,0.244735,0.971566
4,23,4.636406,0.109892,0.234440,0.973608
5,24,4.976217,0.142817,0.230357,0.974679
6,25,5.371387,0.145984,0.226060,0.975577
7,26,5.685317,0.172312,0.209218,0.978897
8,27,6.204570,0.205744,0.214467,0.978066
9,28,6.690865,0.191336,0.202738,0.980256


## 8.8 Directional Propagation

We additionally measure how strongly the downstream perturbation remains
aligned with the original F54316 decoder direction.

A decrease in alignment indicates that later layers progressively transform
the feature-level perturbation rather than simply copying it forward.

In [ ]:
# ============================================================
# F54316 DIRECTIONAL PROPAGATION
# ============================================================

F54316 = 54316

decoder = (
    sae.W_dec[F54316]
    .detach()
    .clone()
    .float()
)

decoder = (
    decoder
    / decoder.norm().clamp_min(1e-8)
)

direction_rows = []

for word in TRACE_WORDS:

    baseline_states = capture_layer_states(
        word,
        intervene=False,
    )

    intervention_states = capture_layer_states(
        word,
        intervene=True,
        fraction=TRACE_FRACTION,
    )

    for layer_idx in TRACE_LAYERS:

        delta = (
            intervention_states[layer_idx]
            - baseline_states[layer_idx]
        )

        # Original F54316 decoder direction
        d = decoder

        # Make dimensions compatible
        d_layer = d.to(
            device=delta.device,
            dtype=delta.dtype,
        )

        cosine_to_decoder = (
            torch.nn.functional.cosine_similarity(
                delta,
                d_layer.unsqueeze(0),
                dim=-1,
            ).item()
        )

        projection = (
            torch.sum(
                delta * d_layer.unsqueeze(0)
            ).item()
        )

        delta_norm = delta.norm().item()

        direction_rows.append({
            "word": word,
            "layer": layer_idx,
            "delta_norm": delta_norm,
            "cosine_to_F54316_decoder": cosine_to_decoder,
            "projection_on_F54316_decoder": projection,
        })

direction_df = pd.DataFrame(
    direction_rows
)

print(
    "Directional trace shape:",
    direction_df.shape
)

display(direction_df.head(20))

Directional trace shape: (130, 5)


,word,layer,delta_norm,cosine_to_F54316_decoder,projection_on_F54316_decoder
0,leopard,19,0.000000,0.000000,0.000000
1,leopard,20,3.794899,0.750188,2.846889
2,leopard,21,4.328071,0.634973,2.748210
3,leopard,22,4.409543,0.484289,2.135493
4,leopard,23,4.597550,0.435370,2.001635
5,leopard,24,4.895154,0.408689,2.000597
6,leopard,25,5.264730,0.371354,1.955081
7,leopard,26,5.540142,0.352008,1.950173
8,leopard,27,6.013272,0.325272,1.955947
9,leopard,28,6.466556,0.296816,1.919375


In [ ]:
# ============================================================
# DIRECTIONAL PROPAGATION SUMMARY
# ============================================================

direction_summary = (
    direction_df
    .groupby("layer")
    .agg(
        mean_delta_norm=("delta_norm", "mean"),
        mean_decoder_cosine=(
            "cosine_to_F54316_decoder",
            "mean"
        ),
        std_decoder_cosine=(
            "cosine_to_F54316_decoder",
            "std"
        ),
        mean_projection=(
            "projection_on_F54316_decoder",
            "mean"
        ),
    )
    .reset_index()
)

print("=== F54316 DIRECTIONAL PROPAGATION ===")
display(direction_summary)

=== F54316 DIRECTIONAL PROPAGATION ===


,layer,mean_delta_norm,mean_decoder_cosine,std_decoder_cosine,mean_projection
0,19,0.000000,0.000000,0.000000,0.000000
1,20,3.735575,0.752616,0.015297,2.810048
2,21,4.261859,0.627277,0.022745,2.672807
3,22,4.372530,0.485045,0.036276,2.120940
4,23,4.636406,0.430632,0.031445,1.997393
5,24,4.976217,0.402869,0.032075,2.005521
6,25,5.371387,0.366324,0.029198,1.968658
7,26,5.685317,0.343938,0.029383,1.956829
8,27,6.204570,0.317811,0.027529,1.972707
9,28,6.690865,0.294804,0.024865,1.973746


# 9. Output-Level Effects

The final stage asks whether the internal F54316 intervention reaches the
model's output distribution.

We measure its effect on the animal-related and vehicle-related continuations.

In [ ]:
# ============================================================
# FINAL LOGIT EFFECT OF F54316
# ============================================================

def continuation_logprob_effect(
    word,
    fraction=0.20,
):
    prompt = make_prompt(word)

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    prefix_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )

    if not torch.is_tensor(prefix_ids):
        prefix_ids = prefix_ids.input_ids

    prefix_ids = prefix_ids.to(model.device)

    prefix_len = prefix_ids.shape[1]
    target_pos = prefix_len - 1

    decoder = (
        sae.W_dec[54316]
        .detach()
        .clone()
        .float()
    )
    decoder = decoder / decoder.norm().clamp_min(1e-8)

    strength = fraction * ref_norm

    def run(intervene):

        def hook(module, inputs, output):

            hidden = (
                output[0]
                if isinstance(output, tuple)
                else output
            )

            modified = hidden.clone()

            if intervene:

                d = decoder.to(
                    device=modified.device,
                    dtype=modified.dtype,
                )

                modified[:, target_pos, :] += (
                    strength * d
                )

            if isinstance(output, tuple):
                return (
                    (modified,)
                    + output[1:]
                )

            return modified

        handle = model.model.layers[
            19
        ].register_forward_hook(hook)

        try:
            with torch.no_grad():
                logits = model(
                    input_ids=prefix_ids,
                    use_cache=False,
                ).logits
        finally:
            handle.remove()

        return logits[0, target_pos].detach().float().cpu()

    baseline_logits = run(False)
    intervention_logits = run(True)

    delta_logits = (
        intervention_logits
        - baseline_logits
    )

    return {
        "baseline": baseline_logits,
        "intervention": intervention_logits,
        "delta": delta_logits,
    }


# ------------------------------------------------------------
# Run on animal-side and vehicle-side behavioral probes
# ------------------------------------------------------------

animal_tokens = tokenizer(
    BEHAVIOR["cont_a"],
    add_special_tokens=False,
)["input_ids"]

vehicle_tokens = tokenizer(
    BEHAVIOR["cont_b"],
    add_special_tokens=False,
)["input_ids"]

print("Animal continuation:", BEHAVIOR["cont_a"])
print("Vehicle continuation:", BEHAVIOR["cont_b"])
print("Animal token IDs:", animal_tokens)
print("Vehicle token IDs:", vehicle_tokens)

Animal continuation:  an animal
Vehicle continuation:  a vehicle
Animal token IDs: [459, 10065]
Vehicle token IDs: [264, 7458]


In [ ]:
# ============================================================
# TOKEN-LEVEL EFFECT
# ============================================================

rows = []

for word in TRACE_WORDS:

    result = continuation_logprob_effect(
        word,
        fraction=0.20,
    )

    delta = result["delta"]

    animal_effect = float(
        delta[
            animal_tokens[0]
        ]
    )

    vehicle_effect = float(
        delta[
            vehicle_tokens[0]
        ]
    )

    rows.append({
        "word": word,
        "animal_token_delta": animal_effect,
        "vehicle_token_delta": vehicle_effect,
        "directional_gap": (
            animal_effect
            - vehicle_effect
        ),
    })

logit_effect_df = pd.DataFrame(rows)

display(logit_effect_df)

,word,animal_token_delta,vehicle_token_delta,directional_gap
0,leopard,0.500000,0.195312,0.304688
1,cheetah,0.218750,-0.156250,0.375000
2,hyena,0.546875,0.343750,0.203125
3,rhinoceros,1.351562,0.140625,1.210938
4,hippopotamus,1.640625,0.281250,1.359375
5,buffalo,0.621094,0.523438,0.097656
6,antelope,2.359375,0.117188,2.242188
7,squirrel,0.226562,-0.406250,0.632812
8,flamingo,0.871094,-0.179688,1.050781
9,meerkat,-0.781250,0.093750,-0.875000


In [ ]:
# ============================================================
# FINAL OUTPUT EFFECT SUMMARY
# ============================================================

print("=== FINAL LOGIT EFFECT ===")

print(
    "Mean animal-token effect:",
    logit_effect_df[
        "animal_token_delta"
    ].mean()
)

print(
    "Mean vehicle-token effect:",
    logit_effect_df[
        "vehicle_token_delta"
    ].mean()
)

print(
    "Mean directional gap:",
    logit_effect_df[
        "directional_gap"
    ].mean()
)

=== FINAL LOGIT EFFECT ===
Mean animal-token effect: 0.75546875
Mean vehicle-token effect: 0.0953125
Mean directional gap: 0.66015625


# 10. Integrated Results

The experiments support the following feature-centered causal pathway:

### Overall Causal Pathway

```text
Layer-18 contributors
        ↓
      F54316
        ↓
Downstream computation
        ↓
   Output logits
        ↓
Animal-vs-vehicle behavior

```
### Main Findings

**Behavior:** The model exhibits a strong animal-vs-vehicle behavioral
dimension.

**Feature:** F54316 generalizes as a strongly animal-associated SAE feature.

**Causality:** Direct intervention on F54316 changes the behavioral margin in
the corresponding direction.

**Upstream mechanism:** Layer-18 Heads 0 and 8 are major contributors to
F54316.

**Mediation:** Restoring F54316 approximately restores behavior after
candidate-head ablation.

**Downstream mechanism:** The perturbation propagates through later layers
while becoming progressively transformed.

**Output:** The intervention preferentially affects animal-related output
probabilities.

# 10. Limitations

This study is a feature-centered case study rather than a complete
reconstruction of the model's animal-related circuit.

Important limitations include the use of one primary feature, one principal
behavioral distinction, relatively small evaluation sets, incomplete
localization of downstream computation, and the absence of a direct
SelfIE-based causal validation.

# 11. Reproducibility

The final research artifact should be accompanied by:

- the Google Colab notebook,
- the exact model and SAE identifiers,
- dependency versions,
- the final numerical results, and
- the code used for causal intervention and circuit tracing.

# 12. Conclusion

This study combines SAE feature characterization, causal activation
intervention, and circuit tracing to examine the relationship between
semantic feature content and model behavior.

For F54316, the evidence supports an animal-related semantic interpretation,
a causal effect on the animal-vs-vehicle behavioral margin, identifiable
upstream contributors, and a measurable downstream effect on
animal-related output logits.